# P2-G4 Dataset-by-Dataset Inspection

- 실행 목적: P2-G4 모델링 착수 전 핵심 데이터셋을 파일별 독립 셀에서 read-only로 검사한다.
- 실행 시각: `2026-07-13T02:54:03.539862+00:00`
- read-only 원칙: 원본 데이터와 readiness 산출물은 수정하지 않고, inspection 프로파일만 `dataset_inspection/`에 저장한다.
- random seed = `3085`
- active D08 hash = `598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962`


In [ ]:

from __future__ import annotations

import hashlib
import io
import json
import os
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
from IPython.display import Markdown, display

RANDOM_STATE = 3085
HEAD_N = 5
TAIL_N = 5
RANDOM_N = 10
TOP_MISSING_N = 30
TOP_CARDINALITY_N = 30

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "workbook").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "workbook").exists():
            PROJECT_ROOT = parent
            break

OUT_ROOT = PROJECT_ROOT / "workbook" / "p2" / "p2_4" / "p4_modeling_readiness_v4"
INSPECTION_ROOT = OUT_ROOT / "dataset_inspection"
PROFILE_DIR = INSPECTION_ROOT / "profiles"
QA_DIR = INSPECTION_ROOT / "qa"
LOG_DIR = INSPECTION_ROOT / "logs"
REPORT_DIR = INSPECTION_ROOT / "reports"
for _path in [PROFILE_DIR, QA_DIR, LOG_DIR, REPORT_DIR]:
    _path.mkdir(parents=True, exist_ok=True)

RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()

try:
    GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
except Exception as exc:
    GIT_COMMIT = f"unknown ({type(exc).__name__})"

print("working directory:", PROJECT_ROOT)
print("Python version:", platform.python_version())
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)
print("pyarrow version:", pyarrow.__version__)
print("platform:", platform.platform())
print("execution time:", RUN_STARTED_AT)
print("Git commit:", GIT_COMMIT)
print("inspection root:", INSPECTION_ROOT)

DATASET_RESULTS = []
ANOMALY_ROWS = []
PROFILE_PATHS = []


working directory: /home/sieg/projects-wsl/SBS_dataScience
Python version: 3.12.3
pandas version: 3.0.3
numpy version: 2.5.0
pyarrow version: 24.0.0
platform: Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
execution time: 2026-07-13T02:54:15.479923+00:00
Git commit: 5b1a3d5
inspection root: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/dataset_inspection


In [ ]:

def rel(path: Path | str) -> str:
    path = Path(path)
    try:
        return path.resolve().relative_to(PROJECT_ROOT).as_posix()
    except Exception:
        return path.as_posix()


def safe_label(label: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in label)


def sha256_file(path: Path | str) -> str | None:
    path = Path(path)
    if not path.exists():
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def summarize_json_structure(obj, depth=0, max_depth=4):
    if depth >= max_depth:
        return type(obj).__name__
    if isinstance(obj, dict):
        return {str(k): summarize_json_structure(v, depth + 1, max_depth) for k, v in obj.items()}
    if isinstance(obj, list):
        return {
            "type": "list",
            "length": len(obj),
            "item_structure": summarize_json_structure(obj[0], depth + 1, max_depth) if obj else None,
        }
    return type(obj).__name__


def load_dataset(path: Path | str):
    path = Path(path)
    ext = path.suffix.lower()
    meta = {"encoding": None, "loader": None, "json_raw_type": None, "json_top_level_keys": None, "json_structure": None, "json_raw_preview": None}
    if ext == ".parquet":
        meta["loader"] = "pd.read_parquet"
        return pd.read_parquet(path), meta
    if ext == ".csv":
        errors = []
        for enc in ["utf-8-sig", "cp949", "euc-kr"]:
            try:
                df = pd.read_csv(path, low_memory=False, encoding=enc)
                meta["encoding"] = enc
                meta["loader"] = "pd.read_csv(low_memory=False)"
                return df, meta
            except UnicodeDecodeError as exc:
                errors.append(f"{enc}: {exc}")
        raise UnicodeDecodeError("csv", b"", 0, 1, " / ".join(errors))
    if ext == ".json":
        obj = json.loads(path.read_text(encoding="utf-8"))
        meta["loader"] = "json.load + pd.json_normalize"
        meta["json_raw_type"] = type(obj).__name__
        meta["json_structure"] = summarize_json_structure(obj)
        meta["json_raw_preview"] = json.dumps(obj, ensure_ascii=False, indent=2, default=str)[:12000]
        if isinstance(obj, dict):
            meta["json_top_level_keys"] = list(obj.keys())
            if "files" in obj and isinstance(obj["files"], list):
                df = pd.json_normalize(obj["files"])
            else:
                df = pd.DataFrame({"key": list(obj.keys()), "value": [json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v for v in obj.values()]})
        elif isinstance(obj, list):
            df = pd.json_normalize(obj)
        else:
            df = pd.DataFrame({"value": [obj]})
        return df, meta
    raise ValueError(f"Unsupported file extension: {ext}")


def capture_df_info(df: pd.DataFrame) -> str:
    buf = io.StringIO()
    df.info(buf=buf, verbose=True, show_counts=True, memory_usage=True)
    return buf.getvalue()


def identifier_candidate(column: str) -> bool:
    text = str(column).lower()
    return any(token in text for token in ["id", "uid", "key", "name", "raw", "label", "sha", "path", "file"])


UNHASHABLE_TYPES = (list, dict, tuple, set, np.ndarray)


def hashable_value(value):
    if isinstance(value, UNHASHABLE_TYPES):
        return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
    return value


def hashable_series(s: pd.Series) -> pd.Series:
    obj = s.astype("object")
    has_unhashable = bool(obj.map(lambda x: isinstance(x, UNHASHABLE_TYPES)).to_numpy(dtype=bool).any())
    if has_unhashable:
        return obj.map(hashable_value)
    return s


def make_hashable_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        out[col] = hashable_series(out[col])
    return out


def build_column_profile(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(df)
    for pos, col in enumerate(df.columns, start=1):
        s = df[col]
        hs = hashable_series(s)
        non_null = int(s.notna().sum())
        missing = int(s.isna().sum())
        nunique = int(hs.nunique(dropna=True))
        vc_all = hs.value_counts(dropna=False, normalize=True)
        rows.append({
            "position": pos,
            "column": col,
            "dtype": str(s.dtype),
            "non_null_n": non_null,
            "missing_n": missing,
            "missing_pct": (missing / n * 100) if n else np.nan,
            "nunique": nunique,
            "unique_ratio": (nunique / non_null) if non_null else np.nan,
            "is_all_null": non_null == 0,
            "is_constant": nunique <= 1,
            "is_identifier_candidate": identifier_candidate(col),
            "top_value_share": float(vc_all.iloc[0]) if n and len(vc_all) else np.nan,
        })
    return pd.DataFrame(rows)


def build_numeric_describe(df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns]
    if not numeric_cols:
        return pd.DataFrame()
    numeric_df = df[numeric_cols].apply(pd.to_numeric, errors="coerce").astype("float64")
    desc = numeric_df.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T.reset_index().rename(columns={"index": "column"})
    rows = []
    n = len(df)
    for col in numeric_cols:
        s = pd.to_numeric(df[col], errors="coerce").astype("float64")
        rows.append({
            "column": col,
            "missing_n": int(s.isna().sum()),
            "missing_pct": float(s.isna().mean() * 100) if n else np.nan,
            "zero_n": int((s == 0).sum()),
            "zero_pct": float((s == 0).mean() * 100) if n else np.nan,
            "negative_n": int((s < 0).sum()),
            "inf_n": int(np.isinf(s).sum()),
            "nunique": int(s.nunique(dropna=True)),
        })
    return desc.merge(pd.DataFrame(rows), on="column", how="left")


def build_categorical_describe(df: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in df.columns if c not in df.select_dtypes(include=[np.number]).columns]
    rows = []
    n = len(df)
    for col in cols:
        s = df[col]
        hs = hashable_series(s)
        vc = hs.value_counts(dropna=True)
        top = vc.index[0] if len(vc) else np.nan
        freq = int(vc.iloc[0]) if len(vc) else 0
        sample_values = [str(x) for x in hs.dropna().drop_duplicates().head(5).tolist()]
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "count": int(s.notna().sum()),
            "missing_n": int(s.isna().sum()),
            "missing_pct": float(s.isna().mean() * 100) if n else np.nan,
            "nunique": int(hs.nunique(dropna=True)),
            "top": top,
            "freq": freq,
            "top_pct": float(freq / n * 100) if n else np.nan,
            "sample_values": " | ".join(sample_values),
        })
    return pd.DataFrame(rows)


def build_datetime_describe(df: pd.DataFrame) -> pd.DataFrame:
    cols = list(df.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns)
    rows = []
    for col in cols:
        s = df[col]
        rows.append({"column": col, "min": s.min(), "max": s.max(), "missing_n": int(s.isna().sum()), "nunique": int(s.nunique(dropna=True))})
    return pd.DataFrame(rows)


def save_profile_outputs(label, column_profile, numeric_desc, categorical_desc, top_missing, duplicates):
    base = safe_label(label)
    paths = []
    for suffix, df in [
        ("column_profile", column_profile),
        ("numeric_describe", numeric_desc),
        ("categorical_describe", categorical_desc),
        ("top_missing", top_missing),
        ("duplicates", duplicates),
    ]:
        path = PROFILE_DIR / f"{base}__{suffix}.csv"
        df.to_csv(path, index=False, encoding="utf-8-sig")
        paths.append(path)
    PROFILE_PATHS.extend(paths)
    return paths


def resolve_candidate_key(df: pd.DataFrame, key_cols):
    alias = {"p4_school_uid": "school_uid", "p4_campus_uid": "campus_uid", "p4_dept_uid": "dept_uid"}
    actual = [alias.get(c, c) for c in key_cols]
    exists = all(c in df.columns for c in actual)
    return actual, exists


def range_audit(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        text = str(col).lower()
        if pd.api.types.is_numeric_dtype(s):
            num = pd.to_numeric(s, errors="coerce").astype("float64")
            rows.append({
                "column": col, "check": "numeric_basic", "nan_n": int(num.isna().sum()),
                "pos_inf_n": int(np.isposinf(num).sum()), "neg_inf_n": int(np.isneginf(num).sum()),
                "negative_n": int((num < 0).sum()), "zero_n": int((num == 0).sum()), "violation_n": 0,
            })
            if text.endswith("_pct"):
                bad = int(((num < 0) | (num > 100)).sum())
                rows.append({"column": col, "check": "pct_0_100", "violation_n": bad})
            if text.endswith("_prop"):
                bad = int(((num < 0) | (num > 1)).sum())
                rows.append({"column": col, "check": "prop_0_1", "violation_n": bad})
            if text.endswith("_n") or text.endswith("count"):
                bad = int((num < 0).sum())
                rows.append({"column": col, "check": "count_nonnegative", "violation_n": bad})
            if "year" in text:
                bad = int(((num < 1900) | (num > 2100)).sum())
                rows.append({"column": col, "check": "year_reasonable_1900_2100", "violation_n": bad})
        elif text.endswith("_flag"):
            allowed = {True, False, 0, 1, "0", "1", "True", "False", "true", "false", np.nan, None, pd.NA}
            non_na = s.dropna().astype(str).str.lower()
            bad = int((~non_na.isin(["true", "false", "0", "1"])).sum())
            rows.append({"column": col, "check": "flag_boolean_or_01", "violation_n": bad})
    return pd.DataFrame(rows)


def inspect_dataset(label, path, expected_shape=None, expected_sha256=None, candidate_keys=None):
    candidate_keys = candidate_keys or []
    abs_path = (PROJECT_ROOT / path).resolve() if not Path(path).is_absolute() else Path(path).resolve()
    warnings = []
    status = "PASS"
    display(Markdown(f"### 5.1 File Identification - `{label}`"))
    exists = abs_path.exists()
    actual_sha = sha256_file(abs_path) if exists else None
    hash_match = (actual_sha == expected_sha256) if expected_sha256 else None
    file_info = pd.DataFrame([{
        "label": label,
        "absolute_path": str(abs_path),
        "relative_path": rel(abs_path),
        "exists": exists,
        "file_extension": abs_path.suffix.lower(),
        "file_size": abs_path.stat().st_size if exists else np.nan,
        "mtime": datetime.fromtimestamp(abs_path.stat().st_mtime).isoformat() if exists else np.nan,
        "actual_SHA256": actual_sha,
        "expected_SHA256": expected_sha256,
        "hash_match": hash_match,
    }])
    display(file_info)
    if not exists:
        status = "FAIL_LOAD"
        warnings.append("file does not exist")
        result = {"label": label, "path": rel(abs_path), "actual_shape": None, "expected_shape": expected_shape, "shape_match": False, "actual_sha256": actual_sha, "expected_sha256": expected_sha256, "hash_match": hash_match, "status": status, "warning_n": len(warnings), "warnings": warnings}
        DATASET_RESULTS.append(result)
        return result
    if expected_sha256 and not hash_match:
        status = "FAIL_HASH"
        warnings.append("SHA256 mismatch")

    try:
        df, load_meta = load_dataset(abs_path)
    except Exception as exc:
        status = "FAIL_LOAD"
        warnings.append(f"load error: {type(exc).__name__}: {exc}")
        result = {"label": label, "path": rel(abs_path), "actual_shape": None, "expected_shape": expected_shape, "shape_match": False, "actual_sha256": actual_sha, "expected_sha256": expected_sha256, "hash_match": hash_match, "status": status, "warning_n": len(warnings), "warnings": warnings}
        DATASET_RESULTS.append(result)
        raise

    if abs_path.suffix.lower() == ".json":
        display(Markdown("### JSON raw structure and flattened table"))
        display(Markdown("JSON structure summary"))
        print(json.dumps(load_meta.get("json_structure"), ensure_ascii=False, indent=2, default=str))
        display(Markdown("JSON raw preview"))
        print(load_meta.get("json_raw_preview"))
        display(Markdown("Flattened table preview"))
        display(df.head(30))

    actual_shape = tuple(df.shape)
    shape_match = (actual_shape == tuple(expected_shape)) if expected_shape else None
    if expected_shape and not shape_match and status == "PASS":
        status = "FAIL_SHAPE"
        warnings.append("shape mismatch")

    display(Markdown("### 5.2 Basic Structure"))
    dup_df_for_basic = make_hashable_frame(df)
    basic_rows = [{
        "shape": actual_shape,
        "expected_shape": expected_shape,
        "shape_match": shape_match,
        "row_count": len(df),
        "column_count": df.shape[1],
        "memory_usage_bytes": int(df.memory_usage(deep=True).sum()),
        "index_dtype": str(df.index.dtype),
        "index_unique": bool(df.index.is_unique),
        "exact_duplicate_row_count": int(dup_df_for_basic.duplicated().sum()),
        "loader": load_meta.get("loader"),
        "encoding": load_meta.get("encoding"),
        "json_raw_type": load_meta.get("json_raw_type"),
        "json_top_level_keys": load_meta.get("json_top_level_keys"),
    }]
    display(pd.DataFrame(basic_rows))

    key_summary_rows = []
    key_dup_frames = []
    for key_cols in candidate_keys:
        actual_key, exists_key = resolve_candidate_key(df, key_cols)
        if exists_key:
            null_key_n = int(df[actual_key].isna().any(axis=1).sum())
            key_frame = make_hashable_frame(df[actual_key])
            dup_mask = key_frame.duplicated(keep=False)
            dup_row_n = int(dup_mask.sum())
            dup_group_n = int(key_frame.loc[dup_mask].drop_duplicates().shape[0]) if dup_row_n else 0
            unique_key_n = int(key_frame.drop_duplicates().shape[0])
            if dup_row_n:
                key_dup_frames.append(df.loc[dup_mask].head(30).copy())
        else:
            null_key_n = dup_row_n = dup_group_n = unique_key_n = np.nan
        key_summary_rows.append({
            "candidate_key_requested": " + ".join(key_cols),
            "candidate_key_actual": " + ".join(actual_key),
            "exists": exists_key,
            "null_key_rows": null_key_n,
            "duplicate_key_rows": dup_row_n,
            "duplicate_key_groups": dup_group_n,
            "unique_key_count": unique_key_n,
        })
        if exists_key and (null_key_n or dup_row_n):
            if status == "PASS":
                status = "PASS_WITH_WARNINGS"
            warnings.append(f"key issue {actual_key}: null={null_key_n}, duplicate_rows={dup_row_n}")
    key_summary = pd.DataFrame(key_summary_rows)
    if not key_summary.empty:
        display(Markdown("#### Candidate Key Check"))
        display(key_summary)

    display(Markdown("### 5.3 Full Column List"))
    column_profile = build_column_profile(df)
    display(column_profile)

    display(Markdown("### 5.4 df.info()"))
    print(capture_df_info(df))

    display(Markdown("### 5.5 Head"))
    display(df.head(HEAD_N))

    display(Markdown("### 5.6 Tail"))
    display(df.tail(TAIL_N))

    display(Markdown("### 5.7 Random Sample"))
    if len(df):
        display(df.sample(n=min(RANDOM_N, len(df)), random_state=RANDOM_STATE))
    else:
        display(Markdown("Empty dataset; no random sample."))

    display(Markdown("### 5.8 Numeric Describe"))
    numeric_desc = build_numeric_describe(df)
    display(numeric_desc if not numeric_desc.empty else pd.DataFrame({"message": ["No numeric columns"]}))

    display(Markdown("### 5.8 Categorical/String/Boolean Describe"))
    categorical_desc = build_categorical_describe(df)
    display(categorical_desc if not categorical_desc.empty else pd.DataFrame({"message": ["No categorical/string/boolean columns"]}))

    display(Markdown("### 5.8 Datetime Describe"))
    datetime_desc = build_datetime_describe(df)
    display(datetime_desc if not datetime_desc.empty else pd.DataFrame({"message": ["No datetime columns"]}))

    display(Markdown("### 5.9 Top Missing Columns"))
    top_missing = column_profile[column_profile["missing_n"] > 0].sort_values(["missing_pct", "missing_n"], ascending=False).head(TOP_MISSING_N).copy()
    top_missing.insert(0, "rank", range(1, len(top_missing) + 1))
    if top_missing.empty:
        display(Markdown("No missing values"))
    else:
        display(top_missing[["rank", "column", "dtype", "missing_n", "missing_pct", "non_null_n", "nunique"]])

    display(Markdown("### 5.10 Top Cardinality Columns"))
    display(column_profile.sort_values(["nunique", "unique_ratio"], ascending=False).head(TOP_CARDINALITY_N)[["column", "dtype", "nunique", "unique_ratio"]])

    display(Markdown("### 5.11 All-null / Constant / Near-constant Columns"))
    all_null = column_profile[column_profile["is_all_null"]].copy()
    constant = column_profile[column_profile["is_constant"]].copy()
    near_constant = column_profile[(column_profile["top_value_share"] >= 0.99) & (~column_profile["is_constant"])].copy()
    display(Markdown("All-null columns"))
    display(all_null[["column", "dtype", "missing_n", "missing_pct"]] if not all_null.empty else pd.DataFrame({"message": ["None"]}))
    display(Markdown("Constant columns"))
    display(constant[["column", "dtype", "nunique", "top_value_share"]] if not constant.empty else pd.DataFrame({"message": ["None"]}))
    display(Markdown("Near-constant columns"))
    display(near_constant[["column", "dtype", "nunique", "top_value_share"]] if not near_constant.empty else pd.DataFrame({"message": ["None"]}))
    if not all_null.empty:
        warnings.append(f"all-null columns: {len(all_null)}")
    if not constant.empty:
        warnings.append(f"constant columns: {len(constant)}")
    if not near_constant.empty:
        warnings.append(f"near-constant columns: {len(near_constant)}")

    display(Markdown("### 5.12 Duplicate Check"))
    dup_df = make_hashable_frame(df)
    exact_dup_mask = dup_df.duplicated(keep=False)
    exact_dup_count = int(dup_df.duplicated().sum())
    exact_dup_group_count = int(dup_df.loc[exact_dup_mask].drop_duplicates().shape[0]) if exact_dup_mask.any() else 0
    key_dup_count = int(pd.to_numeric(key_summary.get("duplicate_key_rows", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not key_summary.empty else 0
    key_dup_group_count = int(pd.to_numeric(key_summary.get("duplicate_key_groups", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not key_summary.empty else 0
    duplicate_summary = pd.DataFrame([{
        "exact_duplicate_row_count": exact_dup_count,
        "exact_duplicate_group_count": exact_dup_group_count,
        "candidate_key_duplicate_row_count": key_dup_count,
        "candidate_key_duplicate_group_count": key_dup_group_count,
    }])
    display(duplicate_summary)
    duplicate_rows = pd.concat(([df.loc[exact_dup_mask].head(30)] if exact_dup_mask.any() else []) + key_dup_frames, ignore_index=True) if (exact_dup_mask.any() or key_dup_frames) else pd.DataFrame()
    display(duplicate_rows if not duplicate_rows.empty else pd.DataFrame({"message": ["No duplicate rows to display"]}))
    if exact_dup_count or key_dup_count:
        warnings.append(f"duplicates: exact={exact_dup_count}, key={key_dup_count}")

    display(Markdown("### 5.13 Range and Basic Outlier Checks"))
    range_df = range_audit(df)
    display(range_df if not range_df.empty else pd.DataFrame({"message": ["No numeric/range checks applicable"]}))
    violations = range_df[pd.to_numeric(range_df.get("violation_n", 0), errors="coerce").fillna(0) > 0].copy() if not range_df.empty else pd.DataFrame()
    display(Markdown("Range violations"))
    display(violations if not violations.empty else pd.DataFrame({"message": ["No range violations"]}))
    if not violations.empty:
        warnings.append(f"range violations: {len(violations)}")

    duplicate_save = duplicate_rows.copy()
    save_profile_outputs(label, column_profile, numeric_desc, categorical_desc, top_missing, duplicate_save)

    for _, row in all_null.iterrows():
        ANOMALY_ROWS.append({"dataset": label, "anomaly_type": "all-null", "column": row["column"], "detail": f"missing_n={row['missing_n']}"})
    for _, row in constant.iterrows():
        ANOMALY_ROWS.append({"dataset": label, "anomaly_type": "constant", "column": row["column"], "detail": f"nunique={row['nunique']}"})
    for _, row in near_constant.iterrows():
        ANOMALY_ROWS.append({"dataset": label, "anomaly_type": "near-constant", "column": row["column"], "detail": f"top_value_share={row['top_value_share']:.6f}"})
    for _, row in violations.iterrows():
        ANOMALY_ROWS.append({"dataset": label, "anomaly_type": "range violation", "column": row["column"], "detail": f"{row['check']} violations={row['violation_n']}"})
    if key_dup_count:
        ANOMALY_ROWS.append({"dataset": label, "anomaly_type": "duplicate key", "column": "", "detail": f"duplicate_key_rows={key_dup_count}"})
    if expected_sha256 and not hash_match:
        ANOMALY_ROWS.append({"dataset": label, "anomaly_type": "hash mismatch", "column": "", "detail": f"actual={actual_sha} expected={expected_sha256}"})
    if expected_shape and not shape_match:
        ANOMALY_ROWS.append({"dataset": label, "anomaly_type": "shape mismatch", "column": "", "detail": f"actual={actual_shape} expected={expected_shape}"})

    if status == "PASS" and warnings:
        status = "PASS_WITH_WARNINGS"
    result = {
        "label": label,
        "path": rel(abs_path),
        "actual_shape": actual_shape,
        "expected_shape": expected_shape,
        "shape_match": shape_match,
        "actual_sha256": actual_sha,
        "expected_sha256": expected_sha256,
        "hash_match": hash_match,
        "status": status,
        "warning_n": len(warnings),
        "warnings": warnings,
        "column_profile_path": rel(PROFILE_DIR / f"{safe_label(label)}__column_profile.csv"),
    }
    DATASET_RESULTS.append(result)
    display(Markdown("### 5.14 Cell Final Status"))
    print("DATASET_STATUS:")
    print(status)
    print("WARNINGS:")
    for warning in warnings:
        print("-", warning)
    if not warnings:
        print("- none")
    return result


def load_by_rel(path: str):
    return load_dataset(PROJECT_ROOT / path)[0]


def extra_d08():
    df = pd.read_parquet(PROJECT_ROOT / "workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet")
    membership = pd.read_parquet(OUT_ROOT / "data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet")
    display(Markdown("### 6.1 D08 Additional Checks"))
    display(Markdown("analysis_year distribution"))
    display(df["analysis_year"].value_counts(dropna=False).rename_axis("analysis_year").reset_index(name="rows"))
    display(Markdown("split distribution via final membership merge"))
    split = df[["outcome_row_id"]].merge(membership[["outcome_row_id", "split"]], on="outcome_row_id", how="left", validate="one_to_one")
    display(split["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="rows"))
    display(Markdown("major_group_7 distribution"))
    display(df["major_group_7"].value_counts(dropna=False).rename_axis("major_group_7").reset_index(name="rows"))
    display(pd.DataFrame([{"school_n": df["school_uid"].nunique(), "campus_n": df["campus_uid"].nunique(), "outcome_row_id_unique_n": df["outcome_row_id"].nunique()}]))
    target_cols = ["a_rate_pct", "health_employment_rate_pct", "graduate_school_progression_rate_pct"]
    display(Markdown("Target describe"))
    display(df[target_cols].describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]).T)
    display(Markdown("Structure match status distribution"))
    for col in ["match_status", "match_method", "row_qa_status"]:
        if col in df:
            display(Markdown(col))
            display(df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="rows"))
    display(Markdown("major7 mapping status distribution"))
    for col in ["major7_mapping_method", "major7_mapping_confidence", "has_major_group_7_high_medium"]:
        if col in df:
            display(Markdown(col))
            display(df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="rows"))
    display(Markdown("Target missing"))
    missing_cols = ["a_rate_pct", "health_employment_rate_pct", "graduate_school_progression_rate_pct", "selectivity_proxy_pct"]
    display(pd.DataFrame([{"column": c, "missing_n": int(df[c].isna().sum()), "missing_pct": float(df[c].isna().mean()*100)} for c in missing_cols]))


def extra_membership():
    df = pd.read_parquet(OUT_ROOT / "data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet")
    d08 = pd.read_parquet(PROJECT_ROOT / "workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet")
    display(Markdown("### 6.2 Sample Membership Additional Checks"))
    rows = []
    for col in [c for c in df.columns if c.startswith("sample_")]:
        s = df[col]
        rows.append({"flag": col, "true_n": int((s == True).sum()), "false_n": int((s == False).sum()), "na_n": int(s.isna().sum())})
    display(pd.DataFrame(rows))
    merged = d08[["outcome_row_id"]].merge(df[["outcome_row_id"]], on="outcome_row_id", how="outer", indicator=True, validate="one_to_one")
    display(merged["_merge"].value_counts(dropna=False).rename_axis("merge_status").reset_index(name="rows"))


def extra_sample_registry():
    reg = pd.read_csv(OUT_ROOT / "artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv", low_memory=False)
    mem = pd.read_parquet(OUT_ROOT / "data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet")
    display(Markdown("### 6.3 Sample Registry Recalculation"))
    rows = []
    for _, r in reg.iterrows():
        sample_id = r["sample_id"]
        col = f"sample_{sample_id}"
        mask = mem[col].fillna(False).astype(bool)
        sub = mem[mask]
        rows.append({
            "sample_id": sample_id,
            "registry_row_n": r["row_n"], "actual_row_n": int(mask.sum()), "diff_row_n": int(mask.sum()) - int(r["row_n"]),
            "registry_school_n": r["school_n"], "actual_school_n": int(sub["school_uid"].nunique()), "diff_school_n": int(sub["school_uid"].nunique()) - int(r["school_n"]),
            "registry_train_n": r["train_n"], "actual_train_n": int(sub["split"].eq("train").sum()),
            "registry_validation_n": r["validation_n"], "actual_validation_n": int(sub["split"].eq("val").sum()),
            "registry_test_n": r["test_n"], "actual_test_n": int(sub["split"].eq("test").sum()),
        })
    display(pd.DataFrame(rows))


def extra_column_registry():
    df = pd.read_csv(OUT_ROOT / "artifacts/department_model_column_registry_v4.csv", low_memory=False)
    display(Markdown("### 6.4 Column Registry Additional Checks"))
    for col in ["semantic_role", "feature_block", "measurement_level", "dtype_actual", "model_default_active", "review_required"]:
        display(Markdown(col))
        display(df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="rows"))
    checks = pd.DataFrame([{
        "OTHER_count": int(df["feature_block"].eq("OTHER").sum()),
        "feature_block_missing": int(df["feature_block"].isna().sum()),
        "dtype_missing": int(df["dtype_actual"].isna().sum()),
        "source_missing": int(df["source_dataset"].isna().sum()),
        "duplicate_column_rows": int(df.duplicated(["column"]).sum()),
    }])
    display(checks)


def extra_feature_set():
    df = pd.read_csv(OUT_ROOT / "artifacts/P4_PHASE_FEATURE_SET_FINAL.csv", low_memory=False)
    colreg = pd.read_csv(OUT_ROOT / "artifacts/department_model_column_registry_v4.csv", low_memory=False)
    display(Markdown("### 6.5 Feature Set Additional Checks"))
    display(df.groupby(["model_id", "target"], dropna=False).agg(feature_count=("feature", "nunique"), allowed_count=("included_in_contract", "sum")).reset_index())
    display(Markdown("Block count by model"))
    display(df.groupby(["model_id", "feature_block"], dropna=False).size().reset_index(name="feature_n"))
    merged = df.merge(colreg[["column", "feature_block", "is_identifier", "is_quality_metadata"]], left_on="feature", right_on="column", how="left", suffixes=("", "_registry"))
    bad = merged[(merged["is_identifier"].fillna(False)) | (merged["is_quality_metadata"].fillna(False)) | merged["feature"].astype(str).str.contains("name|raw|uid|row_id", case=False, regex=True)]
    display(Markdown("Active feature ID/name/QUALITY leakage check"))
    display(bad if not bad.empty else pd.DataFrame({"message": ["No ID/name/QUALITY feature in active feature set"]}))


def extra_model_spec():
    df = pd.read_csv(OUT_ROOT / "artifacts/P4_PHASE_MODEL_SPEC_FINAL.csv", low_memory=False)
    display(Markdown("### 6.6 Model Specification Additional Checks"))
    display(df[["model_id", "sample_id", "target", "allowed_blocks", "feature_count", "status"]])
    display(pd.pivot_table(df, index=["phase", "target"], columns="status", values="model_id", aggfunc="count", fill_value=0).reset_index())


def extra_rank():
    df = pd.read_csv(OUT_ROOT / "qa/P4_PHASE_DESIGN_MATRIX_RANK.csv", low_memory=False)
    display(Markdown("### 6.7 Rank Additional Checks"))
    display(Markdown("rank_deficiency > 0"))
    display(df[pd.to_numeric(df["rank_deficiency"], errors="coerce").fillna(0) > 0])
    display(Markdown("condition_number > 30"))
    display(df[pd.to_numeric(df["condition_number"], errors="coerce").fillna(0) > 30])
    display(Markdown("condition_number > 100"))
    display(df[pd.to_numeric(df["condition_number"], errors="coerce").fillna(0) > 100])


def extra_alias():
    df = pd.read_csv(OUT_ROOT / "qa/P4_LINEAR_ALIAS_GROUPS.csv", low_memory=False)
    display(Markdown("### 6.7 Alias Additional Checks"))
    display(df.groupby(["model_id", "coding", "alias_type"], dropna=False).size().reset_index(name="alias_group_n"))
    display(df[["model_id", "coding", "group_id", "encoded_features", "recommended_exclusion"]].head(80))


def extra_vif():
    df = pd.read_csv(OUT_ROOT / "qa/P4_VIF_TOP30_BY_PHASE.csv", low_memory=False)
    display(Markdown("### 6.7 VIF Additional Checks"))
    bins = []
    for v in df["vif"]:
        if np.isinf(v):
            bins.append("inf")
        elif v < 5:
            bins.append("VIF < 5")
        elif v < 10:
            bins.append("5 <= VIF < 10")
        else:
            bins.append("VIF >= 10")
    df = df.assign(vif_bucket=bins)
    display(df.groupby(["model_id", "vif_bucket"], dropna=False).size().reset_index(name="feature_n"))
    display(df.sort_values(["model_id", "rank"]).groupby("model_id").head(5))


def extra_target_profile():
    df = pd.read_csv(OUT_ROOT / "profiles/P4_TARGET_PROFILE_BY_PHASE.csv", low_memory=False)
    display(Markdown("### 6.8 Target Profile Additional Checks"))
    overall = df[df["row_type"].eq("overall")].copy()
    display(overall[["sample_id", "target", "N", "zero_ratio", "hundred_ratio", "skewness", "school_icc"]])
    display(Markdown("High zero_ratio"))
    display(overall.sort_values("zero_ratio", ascending=False).head(20))
    display(Markdown("High hundred_ratio"))
    display(overall.sort_values("hundred_ratio", ascending=False).head(20))
    display(Markdown("High absolute skewness"))
    display(overall.assign(abs_skewness=overall["skewness"].abs()).sort_values("abs_skewness", ascending=False).head(20))


def extra_nested():
    df = pd.read_csv(OUT_ROOT / "qa/P3_NESTED_CROSSFIT_SMOKE.csv", low_memory=False)
    mem = pd.read_parquet(OUT_ROOT / "data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet")
    total = int(mem["sample_P3_STRUCTURE_READY"].fillna(False).astype(bool).sum())
    display(Markdown("### 6.9 Nested Crossfit Smoke Additional Checks"))
    row = df.iloc[0].to_dict()
    table = pd.DataFrame([{
        "outer_train_N": total - int(row["prediction_n"]),
        "outer_validation_N": int(row["prediction_n"]),
        "train_school_N": int(row["outer_train_schools"]),
        "validation_school_N": int(row["outer_validation_schools"]),
        "selected_alpha": row["selected_alpha"],
        "alpha_grid": "0.1|1.0|10.0|100.0",
        "selected_alpha_is_grid_max": float(row["selected_alpha"]) == 100.0,
        "prediction_N": int(row["prediction_n"]),
        "NaN_prediction_N": int(row["nan_prediction_n"]),
        "encoded_feature_N": int(row["encoded_dimensions"]),
    }])
    display(table)


def extra_bootstrap():
    df = pd.read_csv(OUT_ROOT / "qa/P4_FULL_PIPELINE_BOOTSTRAP_SMOKE.csv", low_memory=False)
    display(Markdown("### 6.10 Bootstrap Smoke Additional Checks"))
    out = df.assign(
        P3_success=df["p3_crossfit_nan_n"].eq(0),
        P4_E_success=df["p4_e_fit_n"].gt(0),
        P4_P_success=df["p4_p_fit_n"].gt(0),
        stored_metric=df["p4_e_rmse_in_boot_smoke"].astype(str) + " | " + df["p4_p_rmse_in_boot_smoke"].astype(str),
    )
    display(out[["iteration", "school_resample_n", "P3_success", "P4_E_success", "P4_P_success", "stored_metric"]])


def extra_final_json():
    path = OUT_ROOT / "reports/P4_FINAL_MODELING_READINESS.json"
    obj = json.loads(path.read_text(encoding="utf-8"))
    display(Markdown("### 6.11 Final Readiness JSON Key-Value Table"))
    display(pd.DataFrame({"key": list(obj.keys()), "value": [json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v for v in obj.values()]}))


def extra_bundle_manifest():
    path = OUT_ROOT / "P4_MODELING_REVIEW_BUNDLE_MANIFEST.json"
    obj = json.loads(path.read_text(encoding="utf-8"))
    display(Markdown("### 6.12 Review Bundle Manifest Files"))
    files = pd.json_normalize(obj.get("files", []))
    cols = [c for c in ["label", "relative_path", "shape", "size_bytes", "sha256", "mtime"] if c in files.columns]
    display(files[cols])


## Dataset: `mart_department_model_base_2024`

- 역할: 2024 modeling single source of truth D08
- 경로: `workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet`
- 예상 shape: `(10242, 151)`
- 예상 SHA256: `598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962`


In [ ]:
result = inspect_dataset(
    label='mart_department_model_base_2024',
    path='workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet',
    expected_shape=(10242, 151),
    expected_sha256='598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962',
    candidate_keys=[['outcome_row_id'], ['analysis_year', 'p4_school_uid', 'p4_campus_uid', 'p4_dept_uid', 'outcome_row_id']],
)

extra_d08()

### 5.1 File Identification - `mart_department_model_base_2024`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,mart_department_model_base_2024,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024...,workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet,True,.parquet,2103297,2026-07-12T21:47:51.507501,598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962,598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(10242, 151)","(10242, 151)",True,10242,151,16773447,int64,True,0,pd.read_parquet,None,None,None


#### Candidate Key Check

...

,major7_mapping_confidence,rows
0,high,8561
1,medium,1538
2,low,85
3,unknown,58


has_major_group_7_high_medium

,has_major_group_7_high_medium,rows
0,True,10099
1,False,143


Target missing

,column,missing_n,missing_pct
0,a_rate_pct,0,0.000000
1,health_employment_rate_pct,2765,26.996680
2,graduate_school_progression_rate_pct,2655,25.922671
3,selectivity_proxy_pct,6505,63.512986


## Dataset: `P4_PHASE_SAMPLE_MEMBERSHIP_FINAL`

- 역할: Final phase-specific sample membership flags
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet`
- 예상 shape: `(10242, 21)`
- 예상 SHA256: `58d0308aa92f04badea3c7fc2db44fc28d0d93580db9bff4f16d47d870ebc7c6`


In [ ]:
result = inspect_dataset(
    label='P4_PHASE_SAMPLE_MEMBERSHIP_FINAL',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet',
    expected_shape=(10242, 21),
    expected_sha256='58d0308aa92f04badea3c7fc2db44fc28d0d93580db9bff4f16d47d870ebc7c6',
    candidate_keys=[['outcome_row_id']],
)

extra_membership()

### 5.1 File Identification - `P4_PHASE_SAMPLE_MEMBERSHIP_FINAL`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_PHASE_SAMPLE_MEMBERSHIP_FINAL,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/data/P4_PHASE_SAMPLE_MEMBERSHIP_FI...,workbook/p2/p2_4/p4_modeling_readiness_v4/data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet,True,.parquet,273814,2026-07-13T11:12:50.209943,58d0308aa92f04badea3c7fc2db44fc28d0d93580db9bff4f16d47d870ebc7c6,58d0308aa92f04badea3c7fc2db44fc28d0d93580db9bff4f16d47d870ebc7c6,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(10242, 21)","(10242, 21)",True,10242,21,1634494,int64,True,0,pd.read_parquet,None,None,None


#### Candidate Key Check

...

### 5.14 Cell Final Status

DATASET_STATUS:
PASS_WITH_WARNINGS
WARNINGS:
- constant columns: 1


### 6.2 Sample Membership Additional Checks

,flag,true_n,false_n,na_n
0,sample_P1_STRUCTURE_READY,6270,3972,0
1,sample_P1_SELECTIVITY_READY,2416,7826,0
2,sample_P2_STRUCTURE_READY,8561,1681,0
3,sample_P2_SELECTIVITY_READY,3198,7044,0
4,sample_P3_STRUCTURE_READY,8561,1681,0
5,sample_P3_SELECTIVITY_READY,3198,7044,0
6,sample_P4_E_STRUCTURE_READY,6270,3972,0
7,sample_P4_P_STRUCTURE_READY,6366,3876,0
8,sample_P4_JOINT_STRUCTURE_READY,6270,3972,0
9,sample_P4_E_SELECTIVITY_READY,2416,7826,0


,merge_status,rows
0,both,10242
1,left_only,0
2,right_only,0


## Dataset: `P4_PHASE_SAMPLE_REGISTRY_FINAL`

- 역할: Final phase sample registry
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv`
- 예상 shape: `(12, 11)`
- 예상 SHA256: `6483fdf67cca469ed7bd2500e5ef917ffd01007194c83461f5c827b03f16805e`


In [ ]:
result = inspect_dataset(
    label='P4_PHASE_SAMPLE_REGISTRY_FINAL',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv',
    expected_shape=(12, 11),
    expected_sha256='6483fdf67cca469ed7bd2500e5ef917ffd01007194c83461f5c827b03f16805e',
    candidate_keys=[['sample_id']],
)

extra_sample_registry()

### 5.1 File Identification - `P4_PHASE_SAMPLE_REGISTRY_FINAL`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_PHASE_SAMPLE_REGISTRY_FINAL,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY...,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv,True,.csv,1587,2026-07-13T11:12:50.205743,6483fdf67cca469ed7bd2500e5ef917ffd01007194c83461f5c827b03f16805e,6483fdf67cca469ed7bd2500e5ef917ffd01007194c83461f5c827b03f16805e,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(12, 11)","(12, 11)",True,12,11,2264,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

,message
0,No range violations


### 5.14 Cell Final Status

DATASET_STATUS:
PASS_WITH_WARNINGS
WARNINGS:
- constant columns: 3


### 6.3 Sample Registry Recalculation

,sample_id,registry_row_n,actual_row_n,diff_row_n,registry_school_n,actual_school_n,diff_school_n,registry_train_n,actual_train_n,registry_validation_n,actual_validation_n,registry_test_n,actual_test_n
0,P1_STRUCTURE_READY,6270,6270,0,186,186,0,4623,4623,932,932,715,715
1,P1_SELECTIVITY_READY,2416,2416,0,130,130,0,1789,1789,367,367,260,260
2,P2_STRUCTURE_READY,8561,8561,0,198,198,0,6302,6302,1263,1263,996,996
3,P2_SELECTIVITY_READY,3198,3198,0,146,146,0,2355,2355,522,522,321,321
4,P3_STRUCTURE_READY,8561,8561,0,198,198,0,6302,6302,1263,1263,996,996
5,P3_SELECTIVITY_READY,3198,3198,0,146,146,0,2355,2355,522,522,321,321
6,P4_E_STRUCTURE_READY,6270,6270,0,186,186,0,4623,4623,932,932,715,715
7,P4_P_STRUCTURE_READY,6366,6366,0,195,195,0,4687,4687,949,949,730,730
8,P4_JOINT_STRUCTURE_READY,6270,6270,0,186,186,0,4623,4623,932,932,715,715
9,P4_E_SELECTIVITY_READY,2416,2416,0,130,130,0,1789,1789,367,367,260,260


## Dataset: `department_model_column_registry_v4`

- 역할: Final column registry with OTHER resolved
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/department_model_column_registry_v4.csv`
- 예상 shape: `(151, 22)`
- 예상 SHA256: `ab75225547d42ddc8e6cdf686be61ecc8bf87512729d8d9d053652b521093b94`


In [ ]:
result = inspect_dataset(
    label='department_model_column_registry_v4',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/department_model_column_registry_v4.csv',
    expected_shape=(151, 22),
    expected_sha256='ab75225547d42ddc8e6cdf686be61ecc8bf87512729d8d9d053652b521093b94',
    candidate_keys=[['column']],
)

extra_column_registry()

### 5.1 File Identification - `department_model_column_registry_v4`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,department_model_column_registry_v4,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/department_model_column_...,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/department_model_column_registry_v4.csv,True,.csv,58300,2026-07-13T11:12:50.014680,ab75225547d42ddc8e6cdf686be61ecc8bf87512729d8d9d053652b521093b94,ab75225547d42ddc8e6cdf686be61ecc8bf87512729d8d9d053652b521093b94,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(151, 22)","(151, 22)",True,151,22,67226,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

model_default_active

,model_default_active,rows
0,False,78
1,True,73


review_required

,review_required,rows
0,False,138
1,True,13


,OTHER_count,feature_block_missing,dtype_missing,source_missing,duplicate_column_rows
0,0,0,0,0,0


## Dataset: `P4_PHASE_FEATURE_SET_FINAL`

- 역할: Final phase-model feature set
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_FEATURE_SET_FINAL.csv`
- 예상 shape: `(250, 11)`
- 예상 SHA256: `dc851803d22a22e891a7fdd9a31dbe0c3d2ac56421a9798e06a9e87e20ce2320`


In [ ]:
result = inspect_dataset(
    label='P4_PHASE_FEATURE_SET_FINAL',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_FEATURE_SET_FINAL.csv',
    expected_shape=(250, 11),
    expected_sha256='dc851803d22a22e891a7fdd9a31dbe0c3d2ac56421a9798e06a9e87e20ce2320',
    candidate_keys=[['model_id', 'feature'], ['phase', 'model_id', 'feature']],
)

extra_feature_set()

### 5.1 File Identification - `P4_PHASE_FEATURE_SET_FINAL`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_PHASE_FEATURE_SET_FINAL,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_FEATURE_SET_FIN...,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_FEATURE_SET_FINAL.csv,True,.csv,39889,2026-07-13T11:12:50.386869,dc851803d22a22e891a7fdd9a31dbe0c3d2ac56421a9798e06a9e87e20ce2320,dc851803d22a22e891a7fdd9a31dbe0c3d2ac56421a9798e06a9e87e20ce2320,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(250, 11)","(250, 11)",True,250,11,56448,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

,model_id,target,feature_count,allowed_count
0,P1_SELECTIVITY,a_rate_pct,17,17
1,P1_STRUCTURE,a_rate_pct,13,13
2,P2_Q,a_rate_pct,15,15
3,P2_S,a_rate_pct,11,11
4,P3_Q,a_rate_pct,15,15
5,P3_S,a_rate_pct,11,11
6,P4_E_JOINT_SELECTIVITY_A_RATE,health_employment_rate_pct,16,16
7,P4_E_JOINT_STRUCTURE_A_RATE,health_employment_rate_pct,12,12
8,P4_E_SELECTIVITY_A_RATE,health_employment_rate_pct,16,16
9,P4_E_SELECTIVITY_RESIDUAL,health_employment_rate_pct,16,16


Block count by model

,model_id,feature_block,feature_n
0,P1_SELECTIVITY,B,5
1,P1_SELECTIVITY,EMP,1
2,P1_SELECTIVITY,POLICY,1
3,P1_SELECTIVITY,PROG,1
4,P1_SELECTIVITY,Q,4
5,P1_SELECTIVITY,S0,5
6,P1_STRUCTURE,B,5
7,P1_STRUCTURE,EMP,1
8,P1_STRUCTURE,POLICY,1
9,P1_STRUCTURE,PROG,1


Active feature ID/name/QUALITY leakage check

,message
0,No ID/name/QUALITY feature in active feature set


## Dataset: `P4_PHASE_MODEL_SPEC_FINAL`

- 역할: Final phase model specification registry
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_MODEL_SPEC_FINAL.csv`
- 예상 shape: `(18, 10)`
- 예상 SHA256: `d7f560167ec752938b73dbee3c5f9dddc84fbc009d9eeaa3905b20946cc7dc48`


In [ ]:
result = inspect_dataset(
    label='P4_PHASE_MODEL_SPEC_FINAL',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_MODEL_SPEC_FINAL.csv',
    expected_shape=(18, 10),
    expected_sha256='d7f560167ec752938b73dbee3c5f9dddc84fbc009d9eeaa3905b20946cc7dc48',
    candidate_keys=[['model_id']],
)

extra_model_spec()

### 5.1 File Identification - `P4_PHASE_MODEL_SPEC_FINAL`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_PHASE_MODEL_SPEC_FINAL,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_MODEL_SPEC_FINA...,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_MODEL_SPEC_FINAL.csv,True,.csv,2618,2026-07-13T11:12:50.388366,d7f560167ec752938b73dbee3c5f9dddc84fbc009d9eeaa3905b20946cc7dc48,d7f560167ec752938b73dbee3c5f9dddc84fbc009d9eeaa3905b20946cc7dc48,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(18, 10)","(18, 10)",True,18,10,3849,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

### 5.14 Cell Final Status

DATASET_STATUS:
PASS
WARNINGS:
- none


### 6.6 Model Specification Additional Checks

,model_id,sample_id,target,allowed_blocks,feature_count,status
0,P1_STRUCTURE,P1_STRUCTURE_READY,a_rate_pct,B+POLICY+S0,13,READY
1,P1_SELECTIVITY,P1_SELECTIVITY_READY,a_rate_pct,B+POLICY+Q+S0,17,READY
2,P2_S,P2_STRUCTURE_READY,a_rate_pct,B+POLICY+S0,11,READY
3,P2_Q,P2_SELECTIVITY_READY,a_rate_pct,B+POLICY+Q+S0,15,READY
4,P3_S,P3_STRUCTURE_READY,a_rate_pct,B+POLICY+S0,11,READY
5,P3_Q,P3_SELECTIVITY_READY,a_rate_pct,B+POLICY+Q+S0,15,READY
6,P4_E_STRUCTURE_A_RATE,P4_E_STRUCTURE_READY,health_employment_rate_pct,B+POLICY+S0,12,READY
7,P4_E_STRUCTURE_RESIDUAL,P4_E_STRUCTURE_READY,health_employment_rate_pct,B+POLICY+S0,12,PENDING_UPSTREAM_RESIDUAL
8,P4_E_SELECTIVITY_A_RATE,P4_E_SELECTIVITY_READY,health_employment_rate_pct,B+POLICY+Q+S0,16,READY
9,P4_E_SELECTIVITY_RESIDUAL,P4_E_SELECTIVITY_READY,health_employment_rate_pct,B+POLICY+Q+S0,16,PENDING_UPSTREAM_RESIDUAL


status,phase,target,PENDING_UPSTREAM_RESIDUAL,READY
0,P1,a_rate_pct,0,2
1,P2,a_rate_pct,0,2
2,P3,a_rate_pct,0,2
3,P4_E,health_employment_rate_pct,2,4
4,P4_P,graduate_school_progression_rate_pct,2,4


## Dataset: `P4_MODEL_FAMILY_RECOMMENDATION`

- 역할: Recommended model family table
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_MODEL_FAMILY_RECOMMENDATION.csv`
- 예상 shape: `(8, 8)`
- 예상 SHA256: `193db8e4da1201fa9ef8fbe635803de560d563e4c1446770129066fbc497c68c`


In [ ]:
result = inspect_dataset(
    label='P4_MODEL_FAMILY_RECOMMENDATION',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_MODEL_FAMILY_RECOMMENDATION.csv',
    expected_shape=(8, 8),
    expected_sha256='193db8e4da1201fa9ef8fbe635803de560d563e4c1446770129066fbc497c68c',
    candidate_keys=[['target_family', 'model_family']],
)

### 5.1 File Identification - `P4_MODEL_FAMILY_RECOMMENDATION`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_MODEL_FAMILY_RECOMMENDATION,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_MODEL_FAMILY_RECOMMEN...,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_MODEL_FAMILY_RECOMMENDATION.csv,True,.csv,2305,2026-07-13T11:12:59.630755,193db8e4da1201fa9ef8fbe635803de560d563e4c1446770129066fbc497c68c,193db8e4da1201fa9ef8fbe635803de560d563e4c1446770129066fbc497c68c,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(8, 8)","(8, 8)",True,8,8,2664,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

,column,check,nan_n,pos_inf_n,neg_inf_n,negative_n,zero_n,violation_n
0,feasible,numeric_basic,0,0,0,0,0,0


Range violations

,message
0,No range violations


### 5.14 Cell Final Status

DATASET_STATUS:
PASS_WITH_WARNINGS
WARNINGS:
- constant columns: 1


## Dataset: `P4_DUPLICATE_CONFLICT_RESOLUTION`

- 역할: Duplicate and conflict handling contract
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_DUPLICATE_CONFLICT_RESOLUTION.csv`
- 예상 shape: `(36, 14)`
- 예상 SHA256: `24d2e000661b3e67a353877383c7ef68b8e156f52bd11ac22f4b1fd724be8d4f`


In [ ]:
result = inspect_dataset(
    label='P4_DUPLICATE_CONFLICT_RESOLUTION',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_DUPLICATE_CONFLICT_RESOLUTION.csv',
    expected_shape=(36, 14),
    expected_sha256='24d2e000661b3e67a353877383c7ef68b8e156f52bd11ac22f4b1fd724be8d4f',
    candidate_keys=[['group_id', 'outcome_row_id']],
)

### 5.1 File Identification - `P4_DUPLICATE_CONFLICT_RESOLUTION`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_DUPLICATE_CONFLICT_RESOLUTION,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_DUPLICATE_CONFLICT_RESOLUTIO...,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_DUPLICATE_CONFLICT_RESOLUTION.csv,True,.csv,13216,2026-07-13T11:12:50.049703,24d2e000661b3e67a353877383c7ef68b8e156f52bd11ac22f4b1fd724be8d4f,24d2e000661b3e67a353877383c7ef68b8e156f52bd11ac22f4b1fd724be8d4f,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(36, 14)","(36, 14)",True,36,14,15436,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

,column,check,nan_n,pos_inf_n,neg_inf_n,negative_n,zero_n,violation_n
0,model_allowed,numeric_basic,0,0,0,0,8,0
1,weight_allowed,numeric_basic,0,0,0,0,36,0
2,denominator_allowed,numeric_basic,0,0,0,0,36,0


Range violations

,message
0,No range violations


### 5.14 Cell Final Status

DATASET_STATUS:
PASS_WITH_WARNINGS
WARNINGS:
- constant columns: 2


## Dataset: `P4_PHASE_DESIGN_MATRIX_RANK`

- 역할: Phase design matrix rank audit
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_PHASE_DESIGN_MATRIX_RANK.csv`
- 예상 shape: `(32, 17)`
- 예상 SHA256: `17414355de01f8ecd9170e810b4826886e5825a57191dacc5d43a1f7b8f336ae`


In [ ]:
result = inspect_dataset(
    label='P4_PHASE_DESIGN_MATRIX_RANK',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_PHASE_DESIGN_MATRIX_RANK.csv',
    expected_shape=(32, 17),
    expected_sha256='17414355de01f8ecd9170e810b4826886e5825a57191dacc5d43a1f7b8f336ae',
    candidate_keys=[['model_id', 'coding']],
)

extra_rank()

### 5.1 File Identification - `P4_PHASE_DESIGN_MATRIX_RANK`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_PHASE_DESIGN_MATRIX_RANK,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_PHASE_DESIGN_MATRIX_RANK.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_PHASE_DESIGN_MATRIX_RANK.csv,True,.csv,5830,2026-07-13T11:12:52.229568,17414355de01f8ecd9170e810b4826886e5825a57191dacc5d43a1f7b8f336ae,17414355de01f8ecd9170e810b4826886e5825a57191dacc5d43a1f7b8f336ae,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(32, 17)","(32, 17)",True,32,17,8178,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

,model_id,sample_id,target,coding,status,train_n,raw_feature_count,numeric_feature_count,categorical_feature_count,encoded_feature_count,rank,rank_deficiency,condition_number,constant_columns,duplicate_encoded_columns_n,perfect_alias_groups_n,recommended_action
0,P1_STRUCTURE,P1_STRUCTURE_READY,a_rate_pct,onehot_full,WARN,4623,13,8.000000,5.000000,38.000000,34.000000,4.000000,110.609575,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
2,P1_SELECTIVITY,P1_SELECTIVITY_READY,a_rate_pct,onehot_full,WARN,1789,17,12.000000,5.000000,39.000000,35.000000,4.000000,88.918897,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
4,P2_S,P2_STRUCTURE_READY,a_rate_pct,onehot_full,WARN,6302,11,6.000000,5.000000,37.000000,33.000000,4.000000,137.818430,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
6,P2_Q,P2_SELECTIVITY_READY,a_rate_pct,onehot_full,WARN,2355,15,10.000000,5.000000,38.000000,34.000000,4.000000,93.787695,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
8,P3_S,P3_STRUCTURE_READY,a_rate_pct,onehot_full,WARN,6302,11,6.000000,5.000000,37.000000,33.000000,4.000000,137.818430,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
10,P3_Q,P3_SELECTIVITY_READY,a_rate_pct,onehot_full,WARN,2355,15,10.000000,5.000000,38.000000,34.000000,4.000000,93.787695,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
12,P4_E_STRUCTURE_A_RATE,P4_E_STRUCTURE_READY,health_employment_rate_pct,onehot_full,WARN,4623,12,7.000000,5.000000,37.000000,33.000000,4.000000,110.647285,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
15,P4_E_SELECTIVITY_A_RATE,P4_E_SELECTIVITY_READY,health_employment_rate_pct,onehot_full,WARN,1789,16,11.000000,5.000000,38.000000,34.000000,4.000000,88.898808,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
18,P4_P_STRUCTURE_A_RATE,P4_P_STRUCTURE_READY,graduate_school_progression_rate_pct,onehot_full,WARN,4687,12,7.000000,5.000000,38.000000,34.000000,4.000000,118.621976,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
21,P4_P_SELECTIVITY_A_RATE,P4_P_SELECTIVITY_READY,graduate_school_progression_rate_pct,onehot_full,WARN,1805,16,11.000000,5.000000,39.000000,35.000000,4.000000,89.589282,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.


condition_number > 30

,model_id,sample_id,target,coding,status,train_n,raw_feature_count,numeric_feature_count,categorical_feature_count,encoded_feature_count,rank,rank_deficiency,condition_number,constant_columns,duplicate_encoded_columns_n,perfect_alias_groups_n,recommended_action
0,P1_STRUCTURE,P1_STRUCTURE_READY,a_rate_pct,onehot_full,WARN,4623,13,8.000000,5.000000,38.000000,34.000000,4.000000,110.609575,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
2,P1_SELECTIVITY,P1_SELECTIVITY_READY,a_rate_pct,onehot_full,WARN,1789,17,12.000000,5.000000,39.000000,35.000000,4.000000,88.918897,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
3,P1_SELECTIVITY,P1_SELECTIVITY_READY,a_rate_pct,treatment,PASS,1789,17,12.000000,5.000000,34.000000,34.000000,0.000000,51.290510,NaN,0.000000,0.000000,NaN
4,P2_S,P2_STRUCTURE_READY,a_rate_pct,onehot_full,WARN,6302,11,6.000000,5.000000,37.000000,33.000000,4.000000,137.818430,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
5,P2_S,P2_STRUCTURE_READY,a_rate_pct,treatment,PASS,6302,11,6.000000,5.000000,32.000000,32.000000,0.000000,99.130561,NaN,0.000000,0.000000,NaN
6,P2_Q,P2_SELECTIVITY_READY,a_rate_pct,onehot_full,WARN,2355,15,10.000000,5.000000,38.000000,34.000000,4.000000,93.787695,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
7,P2_Q,P2_SELECTIVITY_READY,a_rate_pct,treatment,PASS,2355,15,10.000000,5.000000,33.000000,33.000000,0.000000,76.946289,NaN,0.000000,0.000000,NaN
8,P3_S,P3_STRUCTURE_READY,a_rate_pct,onehot_full,WARN,6302,11,6.000000,5.000000,37.000000,33.000000,4.000000,137.818430,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
9,P3_S,P3_STRUCTURE_READY,a_rate_pct,treatment,PASS,6302,11,6.000000,5.000000,32.000000,32.000000,0.000000,99.130561,NaN,0.000000,0.000000,NaN
10,P3_Q,P3_SELECTIVITY_READY,a_rate_pct,onehot_full,WARN,2355,15,10.000000,5.000000,38.000000,34.000000,4.000000,93.787695,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.


condition_number > 100

,model_id,sample_id,target,coding,status,train_n,raw_feature_count,numeric_feature_count,categorical_feature_count,encoded_feature_count,rank,rank_deficiency,condition_number,constant_columns,duplicate_encoded_columns_n,perfect_alias_groups_n,recommended_action
0,P1_STRUCTURE,P1_STRUCTURE_READY,a_rate_pct,onehot_full,WARN,4623,13,8.000000,5.000000,38.000000,34.000000,4.000000,110.609575,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
4,P2_S,P2_STRUCTURE_READY,a_rate_pct,onehot_full,WARN,6302,11,6.000000,5.000000,37.000000,33.000000,4.000000,137.818430,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
8,P3_S,P3_STRUCTURE_READY,a_rate_pct,onehot_full,WARN,6302,11,6.000000,5.000000,37.000000,33.000000,4.000000,137.818430,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
12,P4_E_STRUCTURE_A_RATE,P4_E_STRUCTURE_READY,health_employment_rate_pct,onehot_full,WARN,4623,12,7.000000,5.000000,37.000000,33.000000,4.000000,110.647285,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
18,P4_P_STRUCTURE_A_RATE,P4_P_STRUCTURE_READY,graduate_school_progression_rate_pct,onehot_full,WARN,4687,12,7.000000,5.000000,38.000000,34.000000,4.000000,118.621976,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
24,P4_E_JOINT_STRUCTURE_A_RATE,P4_JOINT_STRUCTURE_READY,health_employment_rate_pct,onehot_full,WARN,4623,12,7.000000,5.000000,37.000000,33.000000,4.000000,110.647285,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.
26,P4_P_JOINT_STRUCTURE_A_RATE,P4_JOINT_STRUCTURE_READY,graduate_school_progression_rate_pct,onehot_full,WARN,4623,12,7.000000,5.000000,37.000000,33.000000,4.000000,110.647285,degree_course_대학과정,0.000000,4.000000,Use treatment coding for linear models.


## Dataset: `P4_LINEAR_ALIAS_GROUPS`

- 역할: Linear model alias group audit
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_LINEAR_ALIAS_GROUPS.csv`
- 예상 shape: `(56, 6)`
- 예상 SHA256: `31340edd978ec94763e6890bda4fef7d63eb414727a30015a054bc7898e59517`


In [ ]:
result = inspect_dataset(
    label='P4_LINEAR_ALIAS_GROUPS',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_LINEAR_ALIAS_GROUPS.csv',
    expected_shape=(56, 6),
    expected_sha256='31340edd978ec94763e6890bda4fef7d63eb414727a30015a054bc7898e59517',
    candidate_keys=[['group_id']],
)

extra_alias()

### 5.1 File Identification - `P4_LINEAR_ALIAS_GROUPS`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_LINEAR_ALIAS_GROUPS,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_LINEAR_ALIAS_GROUPS.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_LINEAR_ALIAS_GROUPS.csv,True,.csv,18248,2026-07-13T11:12:52.232192,31340edd978ec94763e6890bda4fef7d63eb414727a30015a054bc7898e59517,31340edd978ec94763e6890bda4fef7d63eb414727a30015a054bc7898e59517,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(56, 6)","(56, 6)",True,56,6,20654,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

### 5.14 Cell Final Status

DATASET_STATUS:
PASS_WITH_WARNINGS
WARNINGS:
- constant columns: 3


### 6.7 Alias Additional Checks

,model_id,coding,alias_type,alias_group_n
0,P1_SELECTIVITY,onehot_full,svd_nullspace_perfect_alias,4
1,P1_STRUCTURE,onehot_full,svd_nullspace_perfect_alias,4
2,P2_Q,onehot_full,svd_nullspace_perfect_alias,4
3,P2_S,onehot_full,svd_nullspace_perfect_alias,4
4,P3_Q,onehot_full,svd_nullspace_perfect_alias,4
5,P3_S,onehot_full,svd_nullspace_perfect_alias,4
6,P4_E_JOINT_SELECTIVITY_A_RATE,onehot_full,svd_nullspace_perfect_alias,4
7,P4_E_JOINT_STRUCTURE_A_RATE,onehot_full,svd_nullspace_perfect_alias,4
8,P4_E_SELECTIVITY_A_RATE,onehot_full,svd_nullspace_perfect_alias,4
9,P4_E_STRUCTURE_A_RATE,onehot_full,svd_nullspace_perfect_alias,4


,model_id,coding,group_id,encoded_features,recommended_exclusion
0,P1_STRUCTURE,onehot_full,P1_STRUCTURE_onehot_full_SVD_001,major_group_7_MED|major_group_7_ART|major_group_7_EDU|major_group_7_SOC|major_group_7_HUM|major_group_7_NAT|major_gr...,use treatment coding or remove hierarchical source features
1,P1_STRUCTURE,onehot_full,P1_STRUCTURE_onehot_full_SVD_002,campus_branch_|campus_branch_분교|degree_course_대학과정|school_type_산업대학|school_type_대학교|school_type_방송통신대학교|school_type_...,use treatment coding or remove hierarchical source features
2,P1_STRUCTURE,onehot_full,P1_STRUCTURE_onehot_full_SVD_003,degree_course_대학과정|school_type_각종대학(대학)|school_type_방송통신대학교|school_type_교육대학|school_type_산업대학|school_type_대학교|campus...,use treatment coding or remove hierarchical source features
3,P1_STRUCTURE,onehot_full,P1_STRUCTURE_onehot_full_SVD_004,school_type_교육대학|school_type_대학교|school_type_각종대학(대학)|school_type_산업대학|school_type_방송통신대학교|region_sido_대구|region_sid...,use treatment coding or remove hierarchical source features
4,P1_SELECTIVITY,onehot_full,P1_SELECTIVITY_onehot_full_SVD_001,major_group_7_ENG|major_group_7_HUM|major_group_7_SOC|major_group_7_NAT|major_group_7_MED|major_group_7_EDU|major_gr...,use treatment coding or remove hierarchical source features
5,P1_SELECTIVITY,onehot_full,P1_SELECTIVITY_onehot_full_SVD_002,degree_course_대학과정|campus_branch_분교|campus_branch_|school_type_교육대학|school_type_산업대학|school_type_대학교|region_sido_전남|...,use treatment coding or remove hierarchical source features
6,P1_SELECTIVITY,onehot_full,P1_SELECTIVITY_onehot_full_SVD_003,campus_branch_분교|campus_branch_|school_type_교육대학|school_type_산업대학|school_type_대학교|region_sido_경북|region_sido_전북|regi...,use treatment coding or remove hierarchical source features
7,P1_SELECTIVITY,onehot_full,P1_SELECTIVITY_onehot_full_SVD_004,school_type_산업대학|school_type_대학교|school_type_교육대학|degree_course_대학과정|major_group_7_MED|major_group_7_ENG|major_group...,use treatment coding or remove hierarchical source features
8,P2_S,onehot_full,P2_S_onehot_full_SVD_001,major_group_7_EDU|major_group_7_NAT|major_group_7_MED|major_group_7_ENG|major_group_7_ART|major_group_7_HUM|major_gr...,use treatment coding or remove hierarchical source features
9,P2_S,onehot_full,P2_S_onehot_full_SVD_002,school_type_산업대학|school_type_방송통신대학교|school_type_대학교|school_type_교육대학|school_type_각종대학(대학)|degree_course_대학과정|region...,use treatment coding or remove hierarchical source features


## Dataset: `P4_VIF_TOP30_BY_PHASE`

- 역할: Top VIF values by model phase
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_VIF_TOP30_BY_PHASE.csv`
- 예상 shape: `(420, 5)`
- 예상 SHA256: `fdb9ded6ac3dd41eaa3828a34eb671144fe680f3db2c3be094f0c381a6e348ea`


In [ ]:
result = inspect_dataset(
    label='P4_VIF_TOP30_BY_PHASE',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_VIF_TOP30_BY_PHASE.csv',
    expected_shape=(420, 5),
    expected_sha256='fdb9ded6ac3dd41eaa3828a34eb671144fe680f3db2c3be094f0c381a6e348ea',
    candidate_keys=[['model_id', 'rank']],
)

extra_vif()

### 5.1 File Identification - `P4_VIF_TOP30_BY_PHASE`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_VIF_TOP30_BY_PHASE,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_VIF_TOP30_BY_PHASE.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_VIF_TOP30_BY_PHASE.csv,True,.csv,27632,2026-07-13T11:12:52.232858,fdb9ded6ac3dd41eaa3828a34eb671144fe680f3db2c3be094f0c381a6e348ea,fdb9ded6ac3dd41eaa3828a34eb671144fe680f3db2c3be094f0c381a6e348ea,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(420, 5)","(420, 5)",True,420,5,29282,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

### 5.14 Cell Final Status

DATASET_STATUS:
PASS_WITH_WARNINGS
WARNINGS:
- constant columns: 1


### 6.7 VIF Additional Checks

,model_id,vif_bucket,feature_n
0,P1_SELECTIVITY,5 <= VIF < 10,2
1,P1_SELECTIVITY,VIF < 5,28
2,P1_STRUCTURE,VIF < 5,26
3,P1_STRUCTURE,VIF >= 10,4
4,P2_Q,5 <= VIF < 10,1
5,P2_Q,VIF < 5,29
6,P2_S,VIF < 5,26
7,P2_S,VIF >= 10,4
8,P3_Q,5 <= VIF < 10,1
9,P3_Q,VIF < 5,29


,model_id,rank,encoded_feature,vif,corr_matrix_rank_deficient,vif_bucket
30,P1_SELECTIVITY,1,region_sido_서울,6.761274,False,5 <= VIF < 10
31,P1_SELECTIVITY,2,major_group_7_ENG,5.112440,False,5 <= VIF < 10
32,P1_SELECTIVITY,3,school_type_대학교,4.537521,False,VIF < 5
33,P1_SELECTIVITY,4,school_type_산업대학,4.084068,False,VIF < 5
34,P1_SELECTIVITY,5,major_group_7_NAT,3.874281,False,VIF < 5
0,P1_STRUCTURE,1,school_type_대학교,134.631453,False,VIF >= 10
1,P1_STRUCTURE,2,school_type_교육대학,87.177711,False,VIF >= 10
2,P1_STRUCTURE,3,school_type_방송통신대학교,28.997865,False,VIF >= 10
3,P1_STRUCTURE,4,school_type_산업대학,24.094388,False,VIF >= 10
4,P1_STRUCTURE,5,region_sido_서울,4.719472,False,VIF < 5


## Dataset: `P4_TARGET_PROFILE_BY_PHASE`

- 역할: Target distribution profiles by phase sample
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/profiles/P4_TARGET_PROFILE_BY_PHASE.csv`
- 예상 shape: `(396, 20)`
- 예상 SHA256: `6f3fdb25676ccfdf59b0b2f458e5cdb5dd66dfb6f61e037b2e711418efffc25a`


In [ ]:
result = inspect_dataset(
    label='P4_TARGET_PROFILE_BY_PHASE',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/profiles/P4_TARGET_PROFILE_BY_PHASE.csv',
    expected_shape=(396, 20),
    expected_sha256='6f3fdb25676ccfdf59b0b2f458e5cdb5dd66dfb6f61e037b2e711418efffc25a',
    candidate_keys=[['row_type', 'sample_id', 'target', 'group']],
)

extra_target_profile()

### 5.1 File Identification - `P4_TARGET_PROFILE_BY_PHASE`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_TARGET_PROFILE_BY_PHASE,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/profiles/P4_TARGET_PROFILE_BY_PHAS...,workbook/p2/p2_4/p4_modeling_readiness_v4/profiles/P4_TARGET_PROFILE_BY_PHASE.csv,True,.csv,53599,2026-07-13T11:12:59.628858,6f3fdb25676ccfdf59b0b2f458e5cdb5dd66dfb6f61e037b2e711418efffc25a,6f3fdb25676ccfdf59b0b2f458e5cdb5dd66dfb6f61e037b2e711418efffc25a,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(396, 20)","(396, 20)",True,396,20,88704,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

,row_type,sample_id,target,group,N,mean,std,min,p01,p05,p25,median,p75,p95,p99,max,zero_ratio,hundred_ratio,skewness,school_icc
22,overall,P1_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6270,7.636097,12.407968,0.000000,0.000000,0.000000,0.000000,2.074115,10.000000,33.983019,56.030244,92.857140,0.452951,0.000000,2.479303,0.716280
286,overall,P4_JOINT_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6270,7.636097,12.407968,0.000000,0.000000,0.000000,0.000000,2.074115,10.000000,33.983019,56.030244,92.857140,0.452951,0.000000,2.479303,0.716280
220,overall,P4_E_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6270,7.636097,12.407968,0.000000,0.000000,0.000000,0.000000,2.074115,10.000000,33.983019,56.030244,92.857140,0.452951,0.000000,2.479303,0.716280
88,overall,P2_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6366,7.937581,13.136645,0.000000,0.000000,0.000000,0.000000,2.083333,10.460778,35.515706,58.823528,100.000000,0.452403,0.001257,2.641191,1.348605
253,overall,P4_P_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6366,7.937581,13.136645,0.000000,0.000000,0.000000,0.000000,2.083333,10.460778,35.515706,58.823528,100.000000,0.452403,0.001257,2.641191,1.348605
154,overall,P3_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6366,7.937581,13.136645,0.000000,0.000000,0.000000,0.000000,2.083333,10.460778,35.515706,58.823528,100.000000,0.452403,0.001257,2.641191,1.348605
55,overall,P1_SELECTIVITY_READY,graduate_school_progression_rate_pct,ALL,2416,9.115710,12.763240,0.000000,0.000000,0.000000,0.000000,3.571429,13.439354,37.540585,54.156422,72.093025,0.365894,0.000000,1.818274,0.282102
319,overall,P4_E_SELECTIVITY_READY,graduate_school_progression_rate_pct,ALL,2416,9.115710,12.763240,0.000000,0.000000,0.000000,0.000000,3.571429,13.439354,37.540585,54.156422,72.093025,0.365894,0.000000,1.818274,0.282102
385,overall,P4_JOINT_SELECTIVITY_READY,graduate_school_progression_rate_pct,ALL,2416,9.115710,12.763240,0.000000,0.000000,0.000000,0.000000,3.571429,13.439354,37.540585,54.156422,72.093025,0.365894,0.000000,1.818274,0.282102
187,overall,P3_SELECTIVITY_READY,graduate_school_progression_rate_pct,ALL,2439,9.419080,13.311807,0.000000,0.000000,0.000000,0.000000,3.571429,14.052216,39.138961,55.283156,100.000000,0.363264,0.000820,1.942877,0.931104


High hundred_ratio

,row_type,sample_id,target,group,N,mean,std,min,p01,p05,p25,median,p75,p95,p99,max,zero_ratio,hundred_ratio,skewness,school_icc
11,overall,P1_STRUCTURE_READY,health_employment_rate_pct,ALL,6270,52.423729,19.394930,0.000000,0.000000,20.000000,40.638242,52.727272,64.000000,85.714287,100.000000,100.000000,0.019458,0.022967,-0.067152,0.183608
209,overall,P4_E_STRUCTURE_READY,health_employment_rate_pct,ALL,6270,52.423729,19.394930,0.000000,0.000000,20.000000,40.638242,52.727272,64.000000,85.714287,100.000000,100.000000,0.019458,0.022967,-0.067152,0.183608
143,overall,P3_STRUCTURE_READY,health_employment_rate_pct,ALL,6270,52.423729,19.394930,0.000000,0.000000,20.000000,40.638242,52.727272,64.000000,85.714287,100.000000,100.000000,0.019458,0.022967,-0.067152,0.183608
77,overall,P2_STRUCTURE_READY,health_employment_rate_pct,ALL,6270,52.423729,19.394930,0.000000,0.000000,20.000000,40.638242,52.727272,64.000000,85.714287,100.000000,100.000000,0.019458,0.022967,-0.067152,0.183608
275,overall,P4_JOINT_STRUCTURE_READY,health_employment_rate_pct,ALL,6270,52.423729,19.394930,0.000000,0.000000,20.000000,40.638242,52.727272,64.000000,85.714287,100.000000,100.000000,0.019458,0.022967,-0.067152,0.183608
242,overall,P4_P_STRUCTURE_READY,health_employment_rate_pct,ALL,6270,52.423729,19.394930,0.000000,0.000000,20.000000,40.638242,52.727272,64.000000,85.714287,100.000000,100.000000,0.019458,0.022967,-0.067152,0.183608
374,overall,P4_JOINT_SELECTIVITY_READY,health_employment_rate_pct,ALL,2416,53.317039,17.468920,0.000000,9.646962,25.000000,42.539894,53.333332,64.383560,81.638758,100.000000,100.000000,0.006209,0.012831,-0.012148,0.348644
341,overall,P4_P_SELECTIVITY_READY,health_employment_rate_pct,ALL,2416,53.317039,17.468920,0.000000,9.646962,25.000000,42.539894,53.333332,64.383560,81.638758,100.000000,100.000000,0.006209,0.012831,-0.012148,0.348644
308,overall,P4_E_SELECTIVITY_READY,health_employment_rate_pct,ALL,2416,53.317039,17.468920,0.000000,9.646962,25.000000,42.539894,53.333332,64.383560,81.638758,100.000000,100.000000,0.006209,0.012831,-0.012148,0.348644
176,overall,P3_SELECTIVITY_READY,health_employment_rate_pct,ALL,2416,53.317039,17.468920,0.000000,9.646962,25.000000,42.539894,53.333332,64.383560,81.638758,100.000000,100.000000,0.006209,0.012831,-0.012148,0.348644


High absolute skewness

,row_type,sample_id,target,group,N,mean,std,min,p01,p05,p25,median,p75,p95,p99,max,zero_ratio,hundred_ratio,skewness,school_icc,abs_skewness
154,overall,P3_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6366,7.937581,13.136645,0.000000,0.000000,0.000000,0.000000,2.083333,10.460778,35.515706,58.823528,100.000000,0.452403,0.001257,2.641191,1.348605,2.641191
253,overall,P4_P_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6366,7.937581,13.136645,0.000000,0.000000,0.000000,0.000000,2.083333,10.460778,35.515706,58.823528,100.000000,0.452403,0.001257,2.641191,1.348605,2.641191
88,overall,P2_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6366,7.937581,13.136645,0.000000,0.000000,0.000000,0.000000,2.083333,10.460778,35.515706,58.823528,100.000000,0.452403,0.001257,2.641191,1.348605,2.641191
22,overall,P1_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6270,7.636097,12.407968,0.000000,0.000000,0.000000,0.000000,2.074115,10.000000,33.983019,56.030244,92.857140,0.452951,0.000000,2.479303,0.716280,2.479303
286,overall,P4_JOINT_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6270,7.636097,12.407968,0.000000,0.000000,0.000000,0.000000,2.074115,10.000000,33.983019,56.030244,92.857140,0.452951,0.000000,2.479303,0.716280,2.479303
220,overall,P4_E_STRUCTURE_READY,graduate_school_progression_rate_pct,ALL,6270,7.636097,12.407968,0.000000,0.000000,0.000000,0.000000,2.074115,10.000000,33.983019,56.030244,92.857140,0.452951,0.000000,2.479303,0.716280,2.479303
187,overall,P3_SELECTIVITY_READY,graduate_school_progression_rate_pct,ALL,2439,9.419080,13.311807,0.000000,0.000000,0.000000,0.000000,3.571429,14.052216,39.138961,55.283156,100.000000,0.363264,0.000820,1.942877,0.931104,1.942877
352,overall,P4_P_SELECTIVITY_READY,graduate_school_progression_rate_pct,ALL,2439,9.419080,13.311807,0.000000,0.000000,0.000000,0.000000,3.571429,14.052216,39.138961,55.283156,100.000000,0.363264,0.000820,1.942877,0.931104,1.942877
121,overall,P2_SELECTIVITY_READY,graduate_school_progression_rate_pct,ALL,2439,9.419080,13.311807,0.000000,0.000000,0.000000,0.000000,3.571429,14.052216,39.138961,55.283156,100.000000,0.363264,0.000820,1.942877,0.931104,1.942877
55,overall,P1_SELECTIVITY_READY,graduate_school_progression_rate_pct,ALL,2416,9.115710,12.763240,0.000000,0.000000,0.000000,0.000000,3.571429,13.439354,37.540585,54.156422,72.093025,0.365894,0.000000,1.818274,0.282102,1.818274


## Dataset: `P3_NESTED_CROSSFIT_SMOKE`

- 역할: P3 nested cross-fit smoke result
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P3_NESTED_CROSSFIT_SMOKE.csv`
- 예상 shape: `(1, 11)`
- 예상 SHA256: `b784b40c0a8ca7e2d011715594d31d399183539b93303e577ac915c775e25808`


In [ ]:
result = inspect_dataset(
    label='P3_NESTED_CROSSFIT_SMOKE',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P3_NESTED_CROSSFIT_SMOKE.csv',
    expected_shape=(1, 11),
    expected_sha256='b784b40c0a8ca7e2d011715594d31d399183539b93303e577ac915c775e25808',
    candidate_keys=[['status']],
)

extra_nested()

### 5.1 File Identification - `P3_NESTED_CROSSFIT_SMOKE`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P3_NESTED_CROSSFIT_SMOKE,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P3_NESTED_CROSSFIT_SMOKE.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P3_NESTED_CROSSFIT_SMOKE.csv,True,.csv,1097,2026-07-13T11:13:00.305181,b784b40c0a8ca7e2d011715594d31d399183539b93303e577ac915c775e25808,b784b40c0a8ca7e2d011715594d31d399183539b93303e577ac915c775e25808,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(1, 11)","(1, 11)",True,1,11,1008,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

,message
0,No range violations


### 5.14 Cell Final Status

DATASET_STATUS:
PASS_WITH_WARNINGS
WARNINGS:
- constant columns: 11


### 6.9 Nested Crossfit Smoke Additional Checks

,outer_train_N,outer_validation_N,train_school_N,validation_school_N,selected_alpha,alpha_grid,selected_alpha_is_grid_max,prediction_N,NaN_prediction_N,encoded_feature_N
0,6848,1713,158,40,100.000000,0.1|1.0|10.0|100.0,True,1713,0,32


## Dataset: `P4_FULL_PIPELINE_BOOTSTRAP_SMOKE`

- 역할: P4 full pipeline bootstrap smoke result
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_FULL_PIPELINE_BOOTSTRAP_SMOKE.csv`
- 예상 shape: `(3, 11)`
- 예상 SHA256: `5a80872860be75498b84987ebe861e7c17b4180edd6ad4ed2e2cc4acdcbf3467`


In [ ]:
result = inspect_dataset(
    label='P4_FULL_PIPELINE_BOOTSTRAP_SMOKE',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_FULL_PIPELINE_BOOTSTRAP_SMOKE.csv',
    expected_shape=(3, 11),
    expected_sha256='5a80872860be75498b84987ebe861e7c17b4180edd6ad4ed2e2cc4acdcbf3467',
    candidate_keys=[['iteration']],
)

extra_bootstrap()

### 5.1 File Identification - `P4_FULL_PIPELINE_BOOTSTRAP_SMOKE`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_FULL_PIPELINE_BOOTSTRAP_SMOKE,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_FULL_PIPELINE_BOOTSTRAP_SMOK...,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_FULL_PIPELINE_BOOTSTRAP_SMOKE.csv,True,.csv,406,2026-07-13T11:13:04.747216,5a80872860be75498b84987ebe861e7c17b4180edd6ad4ed2e2cc4acdcbf3467,5a80872860be75498b84987ebe861e7c17b4180edd6ad4ed2e2cc4acdcbf3467,True


### 5.2 Basic Structure

,shape,expected_shape,shape_match,row_count,column_count,memory_usage_bytes,index_dtype,index_unique,exact_duplicate_row_count,loader,encoding,json_raw_type,json_top_level_keys
0,"(3, 11)","(3, 11)",True,3,11,408,int64,True,0,pd.read_csv(low_memory=False),utf-8-sig,None,None


#### Candidate Key Check

...

,message
0,No range violations


### 5.14 Cell Final Status

DATASET_STATUS:
PASS_WITH_WARNINGS
WARNINGS:
- constant columns: 3


### 6.10 Bootstrap Smoke Additional Checks

,iteration,school_resample_n,P3_success,P4_E_success,P4_P_success,stored_metric
0,1,186,True,True,True,17.32400778871686 | 10.421646835894284
1,2,186,True,True,True,17.32241164509775 | 10.099755056451666
2,3,186,True,True,True,17.409184830818283 | 10.820265923116253


## Dataset: `P4_FINAL_MODELING_READINESS`

- 역할: Final modeling readiness status JSON
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/reports/P4_FINAL_MODELING_READINESS.json`
- 예상 shape: `None`
- 예상 SHA256: `8fcd0df2cee25f2b31ba498806ea5956e82e5da74580ce01573aa01c201ffbea`


In [ ]:
result = inspect_dataset(
    label='P4_FINAL_MODELING_READINESS',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/reports/P4_FINAL_MODELING_READINESS.json',
    expected_shape=None,
    expected_sha256='8fcd0df2cee25f2b31ba498806ea5956e82e5da74580ce01573aa01c201ffbea',
    candidate_keys=[['key']],
)

extra_final_json()

### 5.1 File Identification - `P4_FINAL_MODELING_READINESS`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_FINAL_MODELING_READINESS,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/reports/P4_FINAL_MODELING_READINES...,workbook/p2/p2_4/p4_modeling_readiness_v4/reports/P4_FINAL_MODELING_READINESS.json,True,.json,1079,2026-07-13T11:13:04.749244,8fcd0df2cee25f2b31ba498806ea5956e82e5da74580ce01573aa01c201ffbea,8fcd0df2cee25f2b31ba498806ea5956e82e5da74580ce01573aa01c201ffbea,True


### JSON raw structure and flattened table

JSON structure summary

{
  "overall_status": "str",
  "P1_MODELING_STATUS": "str",
  "P2_S_MODELING_STATUS": "str",
  "P2_Q_MODELING_STATUS": "str",
  "P3_S_MODELING_STATUS": "str",
  "P3_Q_MODELING_STATUS": "str",
  "P4_E_STRUCTURE_STATUS": "str",
  "P4_P_STRUCTURE_STATUS": "str",
  "P4_JOINT_STRUCTURE_STATUS": "str",
  "P4_E_SELECTIVITY_STATUS": "str",
  "P4_P_SELECTIVITY_STATUS": "str",
  "P4_JOINT_SELECTIVITY_STATUS": "str",
  "P5_2024_EXPLORATORY_STATUS": "str",
  "P7_STATUS": "str",
  "P8_STATUS": "str",
  "rank_block": "bool",
  "sample_block": "bool",
  "pending_residual_spec_n": "int",
  "created_at": "str",
  "active_d08_sha256": "str",
  "active_d08_shape": {
    "type": "list",
    "length": 2,
    "item_structure": "int"
  },
  "input_file_n": "int",
  "duplicate_conflict_excluded_row_n": "int",
  "OTHER_after_resolution": "int"
}


...

,message
0,No range violations


### 5.14 Cell Final Status

DATASET_STATUS:
PASS
WARNINGS:
- none


### 6.11 Final Readiness JSON Key-Value Table

,key,value
0,overall_status,READY_WITH_WARNINGS
1,P1_MODELING_STATUS,READY_WITH_WARNINGS
2,P2_S_MODELING_STATUS,READY_WITH_WARNINGS
3,P2_Q_MODELING_STATUS,READY_WITH_WARNINGS
4,P3_S_MODELING_STATUS,READY_WITH_WARNINGS
5,P3_Q_MODELING_STATUS,READY_WITH_WARNINGS
6,P4_E_STRUCTURE_STATUS,READY_WITH_WARNINGS
7,P4_P_STRUCTURE_STATUS,READY_WITH_WARNINGS
8,P4_JOINT_STRUCTURE_STATUS,READY_WITH_WARNINGS
9,P4_E_SELECTIVITY_STATUS,READY_WITH_WARNINGS


## Dataset: `P4_MODELING_REVIEW_BUNDLE_MANIFEST`

- 역할: Review bundle manifest JSON
- 경로: `workbook/p2/p2_4/p4_modeling_readiness_v4/P4_MODELING_REVIEW_BUNDLE_MANIFEST.json`
- 예상 shape: `None`
- 예상 SHA256: `None`


In [ ]:
result = inspect_dataset(
    label='P4_MODELING_REVIEW_BUNDLE_MANIFEST',
    path='workbook/p2/p2_4/p4_modeling_readiness_v4/P4_MODELING_REVIEW_BUNDLE_MANIFEST.json',
    expected_shape=None,
    expected_sha256=None,
    candidate_keys=[['label']],
)

extra_bundle_manifest()

### 5.1 File Identification - `P4_MODELING_REVIEW_BUNDLE_MANIFEST`

,label,absolute_path,relative_path,exists,file_extension,file_size,mtime,actual_SHA256,expected_SHA256,hash_match
0,P4_MODELING_REVIEW_BUNDLE_MANIFEST,/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_modeling_readiness_v4/P4_MODELING_REVIEW_BUNDLE_MANIFEST...,workbook/p2/p2_4/p4_modeling_readiness_v4/P4_MODELING_REVIEW_BUNDLE_MANIFEST.json,True,.json,8543,2026-07-13T11:13:04.784825,9afa80278fcec363790fb844c056b78c623ac98f46a5556e0049aae24615da79,None,None


### JSON raw structure and flattened table

JSON structure summary

{
  "created_at": "str",
  "files": {
    "type": "list",
    "length": 15,
    "item_structure": {
      "label": "str",
      "path": "str",
      "relative_path": "str",
      "shape": {
        "type": "list",
        "length": 2,
        "item_structure": "int"
      },
      "row_count": "int",
      "column_count": "int",
      "size_bytes": "int",
      "mtime": "str",
      "sha256": "str"
    }
  }
}


...

,message
0,No range violations


### 5.14 Cell Final Status

DATASET_STATUS:
PASS
WARNINGS:
- none


### 6.12 Review Bundle Manifest Files

,label,relative_path,shape,size_bytes,sha256,mtime
0,mart_department_model_base_2024.parquet,workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet,"[10242, 151]",2103297,598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962,2026-07-12T21:47:51.507501
1,P4_PHASE_SAMPLE_REGISTRY_FINAL.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv,"[12, 11]",1587,6483fdf67cca469ed7bd2500e5ef917ffd01007194c83461f5c827b03f16805e,2026-07-13T11:12:50.205743
2,P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet,workbook/p2/p2_4/p4_modeling_readiness_v4/data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet,"[10242, 21]",273814,58d0308aa92f04badea3c7fc2db44fc28d0d93580db9bff4f16d47d870ebc7c6,2026-07-13T11:12:50.209943
3,department_model_column_registry_v4.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/department_model_column_registry_v4.csv,"[151, 22]",58300,ab75225547d42ddc8e6cdf686be61ecc8bf87512729d8d9d053652b521093b94,2026-07-13T11:12:50.014680
4,P4_PHASE_FEATURE_SET_FINAL.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_FEATURE_SET_FINAL.csv,"[250, 11]",39889,dc851803d22a22e891a7fdd9a31dbe0c3d2ac56421a9798e06a9e87e20ce2320,2026-07-13T11:12:50.386869
5,P4_PHASE_MODEL_SPEC_FINAL.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_MODEL_SPEC_FINAL.csv,"[18, 10]",2618,d7f560167ec752938b73dbee3c5f9dddc84fbc009d9eeaa3905b20946cc7dc48,2026-07-13T11:12:50.388366
6,P4_DUPLICATE_CONFLICT_RESOLUTION.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_DUPLICATE_CONFLICT_RESOLUTION.csv,"[36, 14]",13216,24d2e000661b3e67a353877383c7ef68b8e156f52bd11ac22f4b1fd724be8d4f,2026-07-13T11:12:50.049703
7,P4_PHASE_DESIGN_MATRIX_RANK.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_PHASE_DESIGN_MATRIX_RANK.csv,"[32, 17]",5830,17414355de01f8ecd9170e810b4826886e5825a57191dacc5d43a1f7b8f336ae,2026-07-13T11:12:52.229568
8,P4_LINEAR_ALIAS_GROUPS.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_LINEAR_ALIAS_GROUPS.csv,"[56, 6]",18248,31340edd978ec94763e6890bda4fef7d63eb414727a30015a054bc7898e59517,2026-07-13T11:12:52.232192
9,P4_VIF_TOP30_BY_PHASE.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_VIF_TOP30_BY_PHASE.csv,"[420, 5]",27632,fdb9ded6ac3dd41eaa3828a34eb671144fe680f3db2c3be094f0c381a6e348ea,2026-07-13T11:12:52.232858


## Final Integrated Summary

In [ ]:

display(Markdown("## 7. Final Integrated Summary"))

def simple_md_table(df, max_rows=200):
    if df.empty:
        return "(empty)"
    view = df.head(max_rows).copy()
    cols = list(view.columns)
    def clean(value):
        if isinstance(value, (list, tuple, dict)):
            text = json.dumps(value, ensure_ascii=False)
        elif pd.isna(value):
            text = ""
        else:
            text = str(value)
        return text.replace("|", "\\|").replace("\n", " ")
    lines = [
        "| " + " | ".join(cols) + " |",
        "| " + " | ".join(["---"] * len(cols)) + " |",
    ]
    for _, row in view.iterrows():
        lines.append("| " + " | ".join(clean(row[c]) for c in cols) + " |")
    return "\n".join(lines)

inventory = pd.DataFrame(DATASET_RESULTS)
inventory["actual shape"] = inventory["actual_shape"].astype(str)
inventory["expected shape"] = inventory["expected_shape"].astype(str)
inventory_final = inventory[["label", "path", "actual shape", "expected shape", "shape_match", "actual_sha256", "expected_sha256", "hash_match", "status", "warning_n"]].copy()
inventory_path = QA_DIR / "dataset_inventory_final.csv"
inventory_final.to_csv(inventory_path, index=False, encoding="utf-8-sig")
display(Markdown("### 7.1 Dataset inventory"))
display(inventory_final)

missing_frames = []
for profile_path in PROFILE_DIR.glob("*__column_profile.csv"):
    dataset = profile_path.name.replace("__column_profile.csv", "")
    prof = pd.read_csv(profile_path, encoding="utf-8-sig")
    prof.insert(0, "dataset", dataset)
    missing_frames.append(prof[["dataset", "column", "dtype", "missing_n", "missing_pct"]])
all_missing = pd.concat(missing_frames, ignore_index=True).sort_values(["missing_pct", "missing_n"], ascending=False).head(100)
all_missing_path = QA_DIR / "all_datasets_top_missing.csv"
all_missing.to_csv(all_missing_path, index=False, encoding="utf-8-sig")
display(Markdown("### 7.2 All datasets top missing"))
display(all_missing)

anomalies = pd.DataFrame(ANOMALY_ROWS)
if anomalies.empty:
    anomalies = pd.DataFrame(columns=["dataset", "anomaly_type", "column", "detail"])

rank_path = OUT_ROOT / "qa/P4_PHASE_DESIGN_MATRIX_RANK.csv"
if rank_path.exists():
    rank = pd.read_csv(rank_path, low_memory=False)
    for _, row in rank[pd.to_numeric(rank["rank_deficiency"], errors="coerce").fillna(0) > 0].iterrows():
        anomalies.loc[len(anomalies)] = {"dataset": "P4_PHASE_DESIGN_MATRIX_RANK", "anomaly_type": "rank deficiency", "column": row.get("model_id"), "detail": f"coding={row.get('coding')} rank_deficiency={row.get('rank_deficiency')}"}

vif_path = OUT_ROOT / "qa/P4_VIF_TOP30_BY_PHASE.csv"
if vif_path.exists():
    vif = pd.read_csv(vif_path, low_memory=False)
    for _, row in vif[pd.to_numeric(vif["vif"], errors="coerce").fillna(0) >= 10].iterrows():
        anomalies.loc[len(anomalies)] = {"dataset": "P4_VIF_TOP30_BY_PHASE", "anomaly_type": "high VIF", "column": row.get("encoded_feature"), "detail": f"model_id={row.get('model_id')} vif={row.get('vif')}"}

anomaly_path = QA_DIR / "all_dataset_anomalies.csv"
anomalies.to_csv(anomaly_path, index=False, encoding="utf-8-sig")
display(Markdown("### 7.3 All dataset anomalies"))
display(anomalies)

spec = pd.read_csv(OUT_ROOT / "artifacts/P4_PHASE_MODEL_SPEC_FINAL.csv", low_memory=False)
rank = pd.read_csv(OUT_ROOT / "qa/P4_PHASE_DESIGN_MATRIX_RANK.csv", low_memory=False)
sample = pd.read_csv(OUT_ROOT / "artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv", low_memory=False)
focus_models = ["P1_STRUCTURE", "P1_SELECTIVITY", "P2_S", "P2_Q", "P3_S", "P3_Q", "P4_E_STRUCTURE_A_RATE", "P4_E_SELECTIVITY_A_RATE", "P4_P_STRUCTURE_A_RATE", "P4_P_SELECTIVITY_A_RATE"]
phase_map = spec[spec["model_id"].isin(focus_models)].merge(sample[["sample_id", "row_n", "school_n"]], on="sample_id", how="left")
treatment = rank[rank["coding"].eq("treatment")][["model_id", "raw_feature_count", "encoded_feature_count", "rank_deficiency", "status"]].rename(columns={"status": "rank_status"})
phase_map = phase_map.merge(treatment, on="model_id", how="left")
phase_map = phase_map.assign(dataframe="mart_department_model_base_2024", warning=np.where(pd.to_numeric(phase_map["rank_deficiency"], errors="coerce").fillna(0) > 0, "rank_deficiency", ""))
phase_map = phase_map[["phase", "sample_id", "dataframe", "target", "raw_feature_count", "encoded_feature_count", "row_n", "school_n", "status", "warning"]]
phase_map_path = QA_DIR / "phase_data_map.csv"
phase_map.to_csv(phase_map_path, index=False, encoding="utf-8-sig")
display(Markdown("### 7.4 Phase data map"))
display(phase_map)

report_path = REPORT_DIR / "P4_DATASET_INSPECTION_REPORT.md"
summary_path = REPORT_DIR / "P4_DATASET_INSPECTION_SUMMARY.json"
report = f"""# P4 Dataset Inspection Report

- executed_at: {datetime.now(timezone.utc).isoformat()}
- notebook: `{rel(OUT_ROOT / 'p2_G4_dataset_cell_inspection.ipynb')}`
- dataset_count: {len(inventory_final)}
- fail_count: {int(inventory_final['status'].astype(str).str.startswith('FAIL').sum())}
- warning_dataset_count: {int((inventory_final['warning_n'] > 0).sum())}
- anomaly_rows: {len(anomalies)}

## Dataset Status

{simple_md_table(inventory_final)}

## Phase Data Map

{simple_md_table(phase_map)}
"""
report_path.write_text(report, encoding="utf-8")
summary = {
    "executed_at": datetime.now(timezone.utc).isoformat(),
    "dataset_count": int(len(inventory_final)),
    "fail_count": int(inventory_final["status"].astype(str).str.startswith("FAIL").sum()),
    "warning_dataset_count": int((inventory_final["warning_n"] > 0).sum()),
    "anomaly_rows": int(len(anomalies)),
    "inventory_path": rel(inventory_path),
    "all_missing_path": rel(all_missing_path),
    "anomaly_path": rel(anomaly_path),
    "phase_map_path": rel(phase_map_path),
    "report_path": rel(report_path),
}
profile_files = sorted(PROFILE_DIR.glob("*.csv"))
qa_files = sorted(QA_DIR.glob("*.csv"))
report_files = sorted(REPORT_DIR.glob("*"))
env_path = LOG_DIR / "dataset_inspection_environment.json"
run_manifest_path = LOG_DIR / "dataset_inspection_run_manifest.json"
environment = {
    "working_directory": str(PROJECT_ROOT),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "pyarrow_version": pyarrow.__version__,
    "platform": platform.platform(),
    "git_commit": GIT_COMMIT,
    "random_state": RANDOM_STATE,
    "run_started_at": RUN_STARTED_AT,
    "run_completed_at": summary["executed_at"],
}
env_path.write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8")
output_files = profile_files + qa_files + report_files + [env_path]
output_records = []
for output_path in output_files:
    output_records.append({
        "path": rel(output_path),
        "size_bytes": output_path.stat().st_size if output_path.exists() else None,
        "sha256": sha256_file(output_path),
    })
run_manifest = {
    **summary,
    "profile_file_count": len(profile_files),
    "qa_file_count": len(qa_files),
    "report_file_count": len(report_files),
    "environment_path": rel(env_path),
    "outputs": output_records,
}
run_manifest_path.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
summary["profile_file_count"] = len(profile_files)
summary["qa_file_count"] = len(qa_files)
summary["report_file_count"] = len(report_files)
summary["environment_path"] = rel(env_path)
summary["run_manifest_path"] = rel(run_manifest_path)
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
display(Markdown("### Saved final outputs"))
display(pd.DataFrame([summary]))
print("INSPECTION_STATUS:")
print("PASS" if summary["fail_count"] == 0 else "FAIL")


## 7. Final Integrated Summary

### 7.1 Dataset inventory

,label,path,actual shape,expected shape,shape_match,actual_sha256,expected_sha256,hash_match,status,warning_n
0,mart_department_model_base_2024,workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet,"(10242, 151)","(10242, 151)",True,598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962,598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962,True,PASS_WITH_WARNINGS,3
1,P4_PHASE_SAMPLE_MEMBERSHIP_FINAL,workbook/p2/p2_4/p4_modeling_readiness_v4/data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet,"(10242, 21)","(10242, 21)",True,58d0308aa92f04badea3c7fc2db44fc28d0d93580db9bff4f16d47d870ebc7c6,58d0308aa92f04badea3c7fc2db44fc28d0d93580db9bff4f16d47d870ebc7c6,True,PASS_WITH_WARNINGS,1
2,P4_PHASE_SAMPLE_REGISTRY_FINAL,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv,"(12, 11)","(12, 11)",True,6483fdf67cca469ed7bd2500e5ef917ffd01007194c83461f5c827b03f16805e,6483fdf67cca469ed7bd2500e5ef917ffd01007194c83461f5c827b03f16805e,True,PASS_WITH_WARNINGS,1
3,department_model_column_registry_v4,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/department_model_column_registry_v4.csv,"(151, 22)","(151, 22)",True,ab75225547d42ddc8e6cdf686be61ecc8bf87512729d8d9d053652b521093b94,ab75225547d42ddc8e6cdf686be61ecc8bf87512729d8d9d053652b521093b94,True,PASS_WITH_WARNINGS,1
4,P4_PHASE_FEATURE_SET_FINAL,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_FEATURE_SET_FINAL.csv,"(250, 11)","(250, 11)",True,dc851803d22a22e891a7fdd9a31dbe0c3d2ac56421a9798e06a9e87e20ce2320,dc851803d22a22e891a7fdd9a31dbe0c3d2ac56421a9798e06a9e87e20ce2320,True,PASS_WITH_WARNINGS,1
5,P4_PHASE_MODEL_SPEC_FINAL,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_MODEL_SPEC_FINAL.csv,"(18, 10)","(18, 10)",True,d7f560167ec752938b73dbee3c5f9dddc84fbc009d9eeaa3905b20946cc7dc48,d7f560167ec752938b73dbee3c5f9dddc84fbc009d9eeaa3905b20946cc7dc48,True,PASS,0
6,P4_MODEL_FAMILY_RECOMMENDATION,workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_MODEL_FAMILY_RECOMMENDATION.csv,"(8, 8)","(8, 8)",True,193db8e4da1201fa9ef8fbe635803de560d563e4c1446770129066fbc497c68c,193db8e4da1201fa9ef8fbe635803de560d563e4c1446770129066fbc497c68c,True,PASS_WITH_WARNINGS,1
7,P4_DUPLICATE_CONFLICT_RESOLUTION,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_DUPLICATE_CONFLICT_RESOLUTION.csv,"(36, 14)","(36, 14)",True,24d2e000661b3e67a353877383c7ef68b8e156f52bd11ac22f4b1fd724be8d4f,24d2e000661b3e67a353877383c7ef68b8e156f52bd11ac22f4b1fd724be8d4f,True,PASS_WITH_WARNINGS,1
8,P4_PHASE_DESIGN_MATRIX_RANK,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_PHASE_DESIGN_MATRIX_RANK.csv,"(32, 17)","(32, 17)",True,17414355de01f8ecd9170e810b4826886e5825a57191dacc5d43a1f7b8f336ae,17414355de01f8ecd9170e810b4826886e5825a57191dacc5d43a1f7b8f336ae,True,PASS_WITH_WARNINGS,1
9,P4_LINEAR_ALIAS_GROUPS,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_LINEAR_ALIAS_GROUPS.csv,"(56, 6)","(56, 6)",True,31340edd978ec94763e6890bda4fef7d63eb414727a30015a054bc7898e59517,31340edd978ec94763e6890bda4fef7d63eb414727a30015a054bc7898e59517,True,PASS_WITH_WARNINGS,1


### 7.2 All datasets top missing

,dataset,column,dtype,missing_n,missing_pct
121,mart_department_model_base_2024,ctx24_industry_top3_pct,Float32,10242,100.000000
122,mart_department_model_base_2024,ctx24_industry_hhi,Float32,10242,100.000000
158,P4_TARGET_PROFILE_BY_PHASE,min,float64,360,90.909091
159,P4_TARGET_PROFILE_BY_PHASE,p01,float64,360,90.909091
160,P4_TARGET_PROFILE_BY_PHASE,p05,float64,360,90.909091
161,P4_TARGET_PROFILE_BY_PHASE,p25,float64,360,90.909091
163,P4_TARGET_PROFILE_BY_PHASE,p75,float64,360,90.909091
164,P4_TARGET_PROFILE_BY_PHASE,p95,float64,360,90.909091
165,P4_TARGET_PROFILE_BY_PHASE,p99,float64,360,90.909091
166,P4_TARGET_PROFILE_BY_PHASE,max,float64,360,90.909091


...

### 7.4 Phase data map

,phase,sample_id,dataframe,target,raw_feature_count,encoded_feature_count,row_n,school_n,status,warning
0,P1,P1_STRUCTURE_READY,mart_department_model_base_2024,a_rate_pct,13,33.000000,6270,186,READY,
1,P1,P1_SELECTIVITY_READY,mart_department_model_base_2024,a_rate_pct,17,34.000000,2416,130,READY,
2,P2,P2_STRUCTURE_READY,mart_department_model_base_2024,a_rate_pct,11,32.000000,8561,198,READY,
3,P2,P2_SELECTIVITY_READY,mart_department_model_base_2024,a_rate_pct,15,33.000000,3198,146,READY,
4,P3,P3_STRUCTURE_READY,mart_department_model_base_2024,a_rate_pct,11,32.000000,8561,198,READY,
5,P3,P3_SELECTIVITY_READY,mart_department_model_base_2024,a_rate_pct,15,33.000000,3198,146,READY,
6,P4_E,P4_E_STRUCTURE_READY,mart_department_model_base_2024,health_employment_rate_pct,12,32.000000,6270,186,READY,
7,P4_E,P4_E_SELECTIVITY_READY,mart_department_model_base_2024,health_employment_rate_pct,16,33.000000,2416,130,READY,
8,P4_P,P4_P_STRUCTURE_READY,mart_department_model_base_2024,graduate_school_progression_rate_pct,12,33.000000,6366,195,READY,
9,P4_P,P4_P_SELECTIVITY_READY,mart_department_model_base_2024,graduate_school_progression_rate_pct,16,34.000000,2439,136,READY,


### Saved final outputs

,executed_at,dataset_count,fail_count,warning_dataset_count,anomaly_rows,inventory_path,all_missing_path,anomaly_path,phase_map_path,report_path,profile_file_count,qa_file_count,report_file_count,environment_path,run_manifest_path
0,2026-07-13T02:54:20.425855+00:00,16,0,12,104,workbook/p2/p2_4/p4_modeling_readiness_v4/dataset_inspection/qa/dataset_inventory_final.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/dataset_inspection/qa/all_datasets_top_missing.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/dataset_inspection/qa/all_dataset_anomalies.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/dataset_inspection/qa/phase_data_map.csv,workbook/p2/p2_4/p4_modeling_readiness_v4/dataset_inspection/reports/P4_DATASET_INSPECTION_REPORT.md,80,4,2,workbook/p2/p2_4/p4_modeling_readiness_v4/dataset_inspection/logs/dataset_inspection_environment.json,workbook/p2/p2_4/p4_modeling_readiness_v4/dataset_inspection/logs/dataset_inspection_run_manifest.json


INSPECTION_STATUS:
PASS
